# VCC 2026 · 2 — averaging baseline

This version keeps public effect estimation, guide QC, the existing control profiles, coverage reporting, and raw-count generation. It replaces the transfer fit with a fixed equal-weight average across biological backgrounds.

For each measured source effect, retain its full LFC when `abs(lfc) > 2 * SE`; otherwise use zero. Missing or invalid SE is unknown and excluded from the denominator. Apply this filter **once, before averaging**. This is a working signal filter, not an FDR-adjusted DE call. A significant effect of 1 with SE 0.3 stays 1; it is no longer reduced to 0.1 by a small estimated prior.

Each background contributes at most one source per knockdown, using the existing priority rule when multiple tables cover it. Non-significant observations remain zero votes; missing observations do not dilute the average. There is no fitted source scale, ridge residual, generic-response blend, or MAE-driven LOBO search. The count generator applies the strict **mean per-cell control CPM > 5** responder gate once to both effect directions; above the boundary the multiplier is exactly 1. It does not gate all downstream effects by the knocked-down gene's expression or detection rate. Global **α** is the only fitted effect-scale adjustment, with a documented ±8 numerical cap in count generation.

Existing context profiles still report pooled CPM, arithmetic mean per-cell CPM, detection, dispersion, and program levels. Pooled CPM is used for profile similarity diagnostics; arithmetic mean per-cell CPM is used for the expression gate. Similarity does not change weights in this baseline.

Hold H1 out of source effects, benchmark the fixed average with official `cell-eval2`, and compare only a small global α grid. H1 calibration is not an untouched test. Final predictions include all eligible public sources, including H1. This notebook makes no claim of improved biological scores until that benchmark is run.

The >5 CPM exclusion applies to DE metrics; expression and discrimination metrics can still respond to low-expression genes. Setting their LFCs to zero is the explicit modelling choice requested here. See the [official metric specification](https://github.com/ArcInstitute/cell-eval2/blob/main/docs/vcc2026_metrics/vcc2026-metrics.md). The existing workflow's 400-cell output convention is retained; packaging checks the installed submission contract.

All scripts, logs, models and results use a separate `vcc_2026_2` directory. The original notebook and its artifacts remain available. Cells are unexecuted; no cached scores are carried over.


## 0. Session and inputs
Run setup first. The optional download cell uses your existing bucket. Full public H1 cells, Replogle manifests, and optional functional-neighbour edges retain their existing formats. Install runtime dependencies in this kernel's environment; `requirements-eval.txt` supplies the repository's evaluator pins.


In [ ]:
import os, sys, shutil, subprocess, shlex
from pathlib import Path
V = Path.home() / 'vcc'
ENV = dict(CODE=str(V / 'code' / 'vcc_2026_2'), DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_2'),
           B='gs://vcc2026', PY=sys.executable, WORK=str(V / 'lib' / 'vcc_2026_2'))          # PY = this notebook's Python (your venv)
os.environ.update(ENV)
for d in ('CODE', 'LOGS', 'LIB', 'DATA', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now (from the code folder), stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd "$CODE"\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/<name>.log."""
    session = 'vcc2_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd "$CODE"\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc2_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else 'STOPPED - read the log below')
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

# ---- checks ----
need = ['data/gene_names.csv', 'data/pert_counts.csv', 'data/context_A.h5ad', 'data/context_B.h5ad', 'data/context_C.h5ad',
        'lib/orion/HCT116_pseudobulk.h5ad', 'lib/orion/HEK293T_pseudobulk.h5ad',
        'lib/tcell/TCELL_Stim48hr_effects.h5ad', 'lib/tcell/TCELL_Rest_effects.h5ad', 'lib/tcell/tcell_targets_effects.h5ad',
        'lib/h1/H1_full_2025.h5ad']
missing = [f for f in need if not (V / f).exists()]
print('python:', sys.executable)
print('data files: all present' if not missing else f'MISSING input files: {missing}')
print('tmux:', 'OK' if shutil.which('tmux') else 'MISSING -> in the terminal: sudo apt-get install -y tmux')
for m in ['numpy', 'pandas', 'scipy', 'h5py', 'anndata']:
    try:
        __import__(m)
    except ImportError:
        print(f'package {m} MISSING -> in this cell: %pip install {m if m != "sklearn" else "scikit-learn"}')
# Large biological inputs: set these to your downloaded raw-count files.
H1_CELLS = V / 'lib/h1/H1_full_2025.h5ad'  # existing merged H1 file; used directly by effects, benchmark, and oracle
REPLOGLE_MANIFEST = V / 'lib/replogle/manifest.csv'
NEIGHBOURS = V / 'lib/functional_neighbours.csv'
EVAL_PROFILE = 'vcc2026'
LINES = ['A', 'B', 'C']  # use D/E/F with the final bundle
os.environ['H1_CELLS'] = str(H1_CELLS)
os.environ['REPLOGLE_MANIFEST'] = str(REPLOGLE_MANIFEST)
os.environ['NEIGHBOURS'] = str(NEIGHBOURS)
print('official evaluator:', shutil.which('cell-eval2') or 'MISSING: pip install -r requirements-eval.txt')

# Reuse ONLY unfiltered effects and control profiles; never old transfer predictions/settings.
EXISTING_EFFECTS = V / 'lib/review_v4'
REUSE_EXISTING_EFFECTS = True
RUN_NAME = 'vcc_2026_2'
ENV.update(H1_CELLS=str(H1_CELLS), REPLOGLE_MANIFEST=str(REPLOGLE_MANIFEST), NEIGHBOURS=str(NEIGHBOURS))


In [ ]:
DOWNLOAD_INPUTS = False
if DOWNLOAD_INPUTS:
    run(r"""
    mkdir -p "$LIB/orion" "$LIB/tcell" "$LIB/h1"
    gcloud storage cp --no-clobber "$B/data/*" "$DATA/"
    gcloud storage cp --no-clobber "$B/orion/*" "$LIB/orion/"
    gcloud storage cp --no-clobber "$B/tcell/*" "$LIB/tcell/"
    gcloud storage cp --no-clobber "$B/h1/*" "$LIB/h1/"
    """)


## 1. Write the implementation
Run all five script cells after editing. They write only under `~/vcc/code/vcc_2026_2/`.

The effect builder keeps the current label estimators and 0.5 CPM modelling pseudocount. Aggregate Orion and precomputed T-cell labels retain their recorded approximations; they are not relabelled as exact official estimates. The z filter operates on these unfiltered LFC/SE tables. Full H1 and raw Replogle inputs retain guide QC before coverage.

The Replogle manifest uses `cells,name,background,obs_col,control,guide_col,gene_col,layer,platform,kind,priority,target_split`; the first five columns are required. Specify exact non-targeting labels and raw-count layers. K562 releases share background `K562`; the two T-cell states share `CD4_T`.

Optional neighbour edges use `gene_a,gene_b,score`. They fill only targets without any direct source. Filter each measured donor once, mask its own knockdown column, then average available donors by graph score. Borrowed effects are never filtered again and are identified in prediction reports.


### vcc2_common.py


In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_common.py
"""Shared helpers for the VCC 2026 averaging pipeline.

Imported by:
  vcc2_context_profile.py : read_names, alignment, select_controls, moments, save_profile
  vcc2_transfer.py  : Effects, PROGRAMS, load_profile, read_names, save_json, seed_for, significance_filter
  vcc2_build_effects.py   : everything below (writes the formats the other two read)

On-disk formats (one folder per public context, written by vcc2_build_effects.py)
  <dir>/lfc.npy              knockdowns x genes, float16 (memory-mapped), log2 fold change vs that context's controls
  <dir>/se.npy               same shape, standard error of the LFC (NaN = unknown)
  <dir>/targets.npy          knockdown names (row order)
  <dir>/genes.npy            gene names (column order)
  <dir>/meta.json            how the numbers were made (se_kind, lfc definition, legacy flag, ...)
  <dir>/control_profile.npz  baseline profile of the context's controls (same format as a profile .npz)
  <dir>/context.json         suggested name / background / platform for contexts.json
  A context is "legacy" when its SE is missing (no honest uncertainty); loading it then needs allow_legacy=True.

Profile .npz (written by save_profile): genes, meta (json string), and per-gene arrays aligned to `genes`:
  mean_cpm   mean of per-cell CPM (the scorer's > 5 CPM gene gate uses this)
  pooled_cpm ratio-of-sums CPM (sum of counts / total * 1e6), comparable across single-cell AND pseudobulk contexts
  log_mean   log1p(pooled_cpm)  <- the baseline the transfer model compares across contexts
  detection, var_cpm, se_mean (NaN when not computable, e.g. pseudobulk-only contexts)
  plus program_names / program_levels (mean log1p pooled CPM over each program's genes), n_cells, legacy_baseline.
  Genes a context did not measure are NaN.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

CONTROL = "non-targeting"

# Program gene sets. 'prefix:' entries are expanded against the gene list (ribosomal proteins, mitochondrial genes).
PROGRAMS = {
    "cc_S": "MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8".split(),
    "cc_G2M": "HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 CCNB2 CKAP2L "
              "CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 NCAPD2 DLGAP5 "
              "CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA".split(),
    "stress": "HSPA1A HSPA1B HSPA8 HSPH1 DNAJB1 HSP90AA1 FOS JUN EGR1 ATF3 IER2 DUSP1 DDIT3 ATF4".split(),
    "interferon": "ISG15 IFI6 IFIT1 IFIT2 IFIT3 MX1 OAS1 IFI44L STAT1 IRF7 IFITM1 BST2".split(),
    "hypoxia_glyc": "VEGFA CA9 BNIP3 PGK1 LDHA SLC2A1 ENO1 ALDOA NDRG1 P4HA1 PFKFB3 HK2".split(),
    "biogenesis": "NPM1 NCL NOP56 NOP58 FBL RRP9 PA2G4 CAD ODC1 DKC1 NOP16 POLR1B".split(),
    "ribosome": ["prefix:RPL", "prefix:RPS"],
    "mito": ["prefix:MT-"],
}
_PREFIX_EXCLUDE = ("RPL3L", "RPS4Y")


# ---------------------------------------------------------------- small utilities
def read_names(path, col):
    """One column of a CSV as a numpy string array (first column if `col` is absent)."""
    df = pd.read_csv(path)
    return np.asarray(df[col] if col in df.columns else df.iloc[:, 0], dtype=str)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=1, default=_json_default))


def seed_for(seed, *keys):
    """Deterministic seed from (seed, keys...): stable across runs and machines (unlike Python's hash())."""
    h = hashlib.sha256(repr((int(seed),) + tuple(str(k) for k in keys)).encode()).digest()
    return int.from_bytes(h[:8], "little")


SE_EPS = 1e-6


def valid_se(se):
    se = np.asarray(se, dtype=float)
    return np.isfinite(se) & (se > SE_EPS)





def significance_filter(lfc, se):
    """One fixed z filter: retain full measured LFC when |LFC| / SE > 2.

    Measured non-hits become zero. Missing, zero, tiny, negative, or nonfinite
    uncertainty is unknown (NaN), excluded from averaging. This is a working
    signal filter, not a multiple-testing-adjusted DE call or a power model.
    """
    lfc = np.asarray(lfc, dtype=float)
    se = np.asarray(se, dtype=float) if se is not None else np.full_like(lfc, np.nan)
    if lfc.shape != se.shape: raise ValueError('LFC and SE shapes must agree')
    known = np.isfinite(lfc) & valid_se(se)
    return np.where(known, np.where(np.abs(lfc) > 2.0 * se, lfc, 0.0), np.nan).astype(np.float32)


def expression_gate(mean_cpm):
    """Strict scorer boundary, using arithmetic mean per-cell CPM in real controls."""
    mean = np.asarray(mean_cpm, dtype=float)
    return np.isfinite(mean) & (mean > 5.0)


def int64_indptr(matrix):
    """CSR with an int64 row pointer.

    An h5ad X that is written once and then grown with append() keeps the dtype of its
    FIRST block's indptr. scipy uses int32 for small blocks, which caps the finished file at
    2**31 stored values (merged H1 has ~3.5e9; a 360k-cell submission is ~2.2e9).
    """
    m = matrix.tocsr()
    m.indptr = m.indptr.astype(np.int64)
    return m


def validate_counts(matrix, label="counts"):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix)
    if not (np.isfinite(values).all() and (values >= 0).all() and (values == np.floor(values)).all()):
        raise ValueError(f"{label}: expected finite, nonnegative integer raw counts")


# ---------------------------------------------------------------- gene alignment and programs
def alignment(source_names, genes):
    """For each of OUR genes, its column in the source (-1 = the source did not measure it)."""
    pos = {g: i for i, g in enumerate(np.asarray(source_names, dtype=str))}
    idx = np.array([pos.get(g, -1) for g in np.asarray(genes, dtype=str)], dtype=np.int64)
    if (idx >= 0).sum() == 0:
        raise ValueError("no genes in common with the provided gene list")
    return idx


def program_gene_index(genes):
    """{program: indices into genes}."""
    genes = np.asarray(genes, dtype=str); pos = {g: i for i, g in enumerate(genes)}; out = {}
    for name, members in PROGRAMS.items():
        idx = []
        for m in members:
            if m.startswith("prefix:"):
                p = m.split(":", 1)[1]
                idx += [i for i, g in enumerate(genes) if g.startswith(p) and not g.startswith(_PREFIX_EXCLUDE)]
            elif m in pos:
                idx.append(pos[m])
        out[name] = np.array(sorted(set(idx)), dtype=np.int64)
    return out


def program_levels(pooled_cpm, genes):
    """Mean log1p(pooled CPM) over each program's measured genes (NaN if < 3 measured)."""
    lv = []
    for name, idx in program_gene_index(genes).items():
        v = np.log1p(pooled_cpm[idx]) if len(idx) else np.array([])
        v = v[np.isfinite(v)]
        lv.append(float(v.mean()) if len(v) >= 3 else np.nan)
    return np.array(list(PROGRAMS)), np.array(lv, dtype=np.float32)


# ---------------------------------------------------------------- profiles from cells
def select_controls(a, obs_col, control=CONTROL, all_controls=False, max_cells=20000, seed=0):
    """Row indices of control cells. If `obs_col` is absent every row is treated as a control (e.g. context_A.h5ad).
    Without all_controls, at most `max_cells` controls (random, seeded) are used."""
    if obs_col in a.obs.columns:
        rows = np.flatnonzero(a.obs[obs_col].astype(str).values == control)
    else:
        rows = np.arange(a.n_obs)
    if len(rows) == 0:
        raise ValueError(f"no rows with {obs_col} == {control!r}")
    if not all_controls and len(rows) > max_cells:
        rows = np.sort(np.random.default_rng(seed).choice(rows, max_cells, replace=False))
    return np.sort(rows)


def _matrix(a, layer):
    if layer is None:
        return a.X
    if layer in a.layers:
        return a.layers[layer]
    raise KeyError(f"layer {layer!r} not found")


def moments(a, rows, idx, layer=None, block=512):
    """Per-gene statistics over `rows` (cells), streamed `block` cells at a time. Library size = sum over OUR genes.
    Returns arrays aligned to our gene list (NaN where the source lacks the gene)."""
    X = _matrix(a, layer); G = len(idx); ok = idx >= 0; cols = idx[ok]
    s_cnt = np.zeros(ok.sum()); s_cpm = np.zeros(ok.sum()); s_cpm2 = np.zeros(ok.sum()); s_det = np.zeros(ok.sum())
    libs = []
    for b in range(0, len(rows), block):
        r = np.asarray(rows[b:b + block])
        M = X[r]
        M = (M.tocsr() if sp.issparse(M) else sp.csr_matrix(np.asarray(M)))[:, cols].astype(np.float64)
        validate_counts(M)
        lib = np.asarray(M.sum(1)).ravel(); libs.append(lib)
        C = sp.diags(1e6 / np.maximum(lib, 1)) @ M
        s_cnt += np.asarray(M.sum(0)).ravel(); s_cpm += np.asarray(C.sum(0)).ravel()
        s_cpm2 += np.asarray(C.multiply(C).sum(0)).ravel(); s_det += np.asarray((M > 0).sum(0)).ravel()
    n = len(rows); lib = np.concatenate(libs) if libs else np.array([0.0])

    def full(v):
        out = np.full(G, np.nan, dtype=np.float32); out[ok] = v; return out
    if n == 0:
        raise ValueError("no cells to profile")
    mean = s_cpm / n; var = np.maximum(s_cpm2 / n - mean ** 2, 0)
    pooled = s_cnt / max(s_cnt.sum(), 1) * 1e6
    stats = dict(mean_cpm=np.where(ok, 0.0, np.nan), pooled_cpm=full(pooled), bulk_cpm=full(pooled), log_mean=full(np.log1p(pooled)),
                 detection=full(s_det / n), var_cpm=full(var), se_mean=full(np.sqrt(var / n)),
                 n_cells=np.int64(n), median_library=np.float64(np.median(lib)), measured=ok.copy(),
                 legacy_baseline=np.bool_(False))
    stats["mean_cpm"][ok] = mean  # float64 preserves the strict 5-CPM boundary
    return stats


def finalise_profile(stats, genes):
    """Add program levels and the scorer gate flag."""
    names, lv = program_levels(stats["pooled_cpm"], genes)
    stats["program_names"] = names; stats["program_levels"] = lv
    stats["scored"] = expression_gate(stats["mean_cpm"])
    return stats


def save_profile(output, genes, stats, meta=None):
    """Write a profile .npz (program levels added if missing)."""
    stats = dict(stats)
    if "pooled_cpm" in stats:
        stats["bulk_cpm"] = np.asarray(stats["pooled_cpm"]).copy()
        stats["baseline_convention"] = np.array("ratio_of_sums_cpm")
    if "program_levels" not in stats:
        stats = finalise_profile(stats, genes)
    out = Path(output); out.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, genes=np.asarray(genes, dtype=str), meta=json.dumps(meta or {}, default=_json_default), **stats)
    return out


def load_profile(path, genes):
    """Profile re-aligned to `genes`. Per-gene arrays: NaN (or False) where missing. Returns a plain dict."""
    z = np.load(path, allow_pickle=False)
    pg = z["genes"].astype(str); src = alignment(pg, genes); ok = src >= 0
    out = {}
    for k in z.files:
        if k in ("genes", "meta"):
            continue
        v = z[k]
        if v.ndim == 1 and len(v) == len(pg) and k not in ("program_names", "program_levels", "baseline_program_levels", "baseline_program_coverage", "control_obs_names"):
            if v.dtype == bool:
                w = np.zeros(len(genes), bool); w[ok] = v[src[ok]]
            else:
                w = np.full(len(genes), np.nan, dtype=np.float64 if k == "mean_cpm" else np.float32); w[ok] = v[src[ok]]
            out[k] = w
        else:
            out[k] = v.item() if v.ndim == 0 else v
    out["genes"] = np.asarray(genes, dtype=str)
    out["meta"] = json.loads(str(z["meta"])) if "meta" in z.files else {}
    out["legacy_baseline"] = bool(out.get("legacy_baseline", False))
    if "program_levels" not in out:
        out["program_names"], out["program_levels"] = program_levels(np.expm1(out["log_mean"]), genes)
    out["program_levels"] = np.nan_to_num(np.asarray(out["program_levels"], dtype=np.float32), nan=0.0)
    for k in ("log_mean", "mean_cpm"):
        if k not in out:
            raise KeyError(f"profile {path} lacks {k}")
    return out


# ---------------------------------------------------------------- effect tables
class Effects:
    """One public context's effect table: LFC and SE per (knockdown, gene), memory-mapped from disk.
    get(kd, gene_idx) -> (lfc, se) float32 arrays for OUR gene indices (NaN = not measured)."""

    def __init__(self, path, genes, allow_legacy=False):
        p = Path(path); self.path = p
        self.meta = json.loads((p / "meta.json").read_text()) if (p / "meta.json").exists() else {}
        self.legacy = bool(self.meta.get("legacy", False))
        if self.legacy and not allow_legacy:
            raise ValueError(f"{p} is a legacy effect table (no usable SE); pass --allow-legacy-effects to use it")
        self.lfc = np.load(p / "lfc.npy", mmap_mode="r")
        self.se = np.load(p / "se.npy", mmap_mode="r") if (p / "se.npy").exists() else None
        self.targets = [str(t) for t in np.load(p / "targets.npy")]
        self.pos = {t: i for i, t in enumerate(self.targets)}
        self.col = alignment(np.load(p / "genes.npy").astype(str), genes)    # our gene -> stored column
        self.n_genes = len(genes)

    def get(self, kd, gi):
        gi = np.asarray(gi, dtype=np.int64).ravel()
        lfc = np.full(len(gi), np.nan, np.float32); se = np.full(len(gi), np.nan, np.float32)
        r = self.pos.get(kd)
        if r is None:
            return lfc, se
        c = self.col[gi]; ok = c >= 0
        if ok.any():
            row = np.asarray(self.lfc[r], dtype=np.float32); lfc[ok] = row[c[ok]]
            if self.se is not None:
                srow = np.asarray(self.se[r], dtype=np.float32); se[ok] = srow[c[ok]]
        return lfc, se

    def __repr__(self):
        return f"Effects({self.path.name}: {len(self.targets):,} knockdowns, legacy={self.legacy})"


def write_effects(out, genes, targets, rows_iter, meta, n_rows):
    """Stream (k, lfc_row, se_row) into an effect-table folder. Rows arrive in `targets` order."""
    from numpy.lib.format import open_memmap
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    L = open_memmap(out / "lfc.npy", mode="w+", dtype=np.float16, shape=(n_rows, len(genes)))
    S = open_memmap(out / "se.npy", mode="w+", dtype=np.float16, shape=(n_rows, len(genes)))
    done = 0
    for k, lfc, se in rows_iter:
        L[k] = np.clip(lfc, -60000, 60000); S[k] = np.clip(se, 0, 60000); done += 1
    L.flush(); S.flush(); del L, S
    np.save(out / "targets.npy", np.asarray(targets, dtype=str)); np.save(out / "genes.npy", np.asarray(genes, dtype=str))
    save_json(out / "meta.json", dict(meta, n_targets=int(n_rows), n_genes=int(len(genes))))
    assert done == n_rows, (done, n_rows)
    return out


### vcc2_build_effects.py


In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_build_effects.py
"""Build unfiltered public inputs from our public data: effect tables, control profiles and contexts.json.

  orion       Orion per-knockdown sums (<line>_pseudobulk.h5ad)  -> effect folder (LFC + cell-level SE) + control profile
  h1          Arc H1 sums (h1_pseudobulk.h5ad, same layout)      -> same as orion (held-out validation context)
  tcell       T-cell effect table (TCELL_<cond>_effects.h5ad)   -> effect folder (LFC + donor-level SE) + control profile
  make-config effect folders                                     -> contexts.json for vcc2_transfer.py

Every effect folder also gets context.json (name / background / platform), so make-config needs no typing.

Colab
  !python vcc2_build_effects.py orion --pseudobulk /content/drive/MyDrive/vcc/orion/HCT116_pseudobulk.h5ad \
        --genes "$DATA/gene_names.csv" --name ORION_HCT116 --background HCT116 --out $LIB/effects/ORION_HCT116
  !python vcc2_build_effects.py tcell --effects $LIB/tcell/effects/TCELL_Stim48hr_effects.h5ad \
        --genes "$DATA/gene_names.csv" --name TCELL_Stim48hr --priority 1 --out $LIB/effects/TCELL_Stim48hr
  !python vcc2_build_effects.py make-config --genes "$DATA/gene_names.csv" --out $LIB/contexts.json \
        --dirs $LIB/effects/ORION_HCT116 $LIB/effects/ORION_HEK293T $LIB/effects/TCELL_Stim48hr $LIB/effects/TCELL_Rest
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import anndata as ad
import pandas as pd
import h5py
import numpy as np
import scipy.sparse as sp

from vcc2_common import CONTROL, alignment, read_names, save_json, save_profile, write_effects, moments, validate_counts

try:
    from anndata.io import read_elem, sparse_dataset
except ImportError:
    from anndata.experimental import read_elem, sparse_dataset

PSEUDO = 1e-9  # evaluator epsilon (cell-eval2 vcc2026); not used for modelling labels
# Modelling labels add this many CPM to both sides of every ratio. With the evaluator's 1e-9,
# a low-count gene with zero counts in a ~100-cell group gets LFC near -30 with a small
# delta-method SE: it can pass a z filter and appear spuriously confident in
# lines where the gene is expressed. The evaluator only scores genes above 5 CPM in controls,
# where 0.5 CPM moves an LFC by at most ~0.13; below that, noise collapses toward zero while
# genuine induction from ~0 CPM survives (e.g. 0 -> 10 CPM is still +4.4).
LABEL_PSEUDOCOUNT = 0.5


def mean_variance(mean, var_of_mean, unit):
    """Variance of a group's mean CPM, floored at Poisson noise plus one pseudo-count.

    `unit` is the CPM one count adds to the group mean (1e6 / group's total counts). An
    all-zero group has zero empirical variance, which made its label look precise; the
    floor (mean + unit) * unit keeps it as uncertain as one unseen count. Overdispersed
    genes have empirical variance above Poisson, so the floor rarely binds for them.
    """
    return np.maximum(var_of_mean, (np.maximum(mean, 0) + unit) * unit)


def label_pseudocount(a):
    pc = float(getattr(a, "pseudocount", LABEL_PSEUDOCOUNT))
    if not pc > 0: raise ValueError("label pseudocount must be positive CPM")
    return pc


def reader(f, key):
    """rows(list of int) -> dense float64 block, for dense or sparse h5ad matrices."""
    node = f[key]
    if isinstance(node, h5py.Group):
        ds = sparse_dataset(node)
        return lambda r: np.asarray(ds[np.asarray(r)].toarray(), dtype=np.float64)
    return lambda r: np.asarray(node[np.asarray(r), :], dtype=np.float64)


def names_of(df, col="target_gene"):
    return np.asarray(df[col] if col in df.columns else df.index, dtype=str)


def context_file(out, name, background, platform, priority, kind):
    save_json(Path(out) / "context.json", dict(name=name, background=background, platform=platform,
                                             source_priority=priority, kind=kind))


# ---------------------------------------------------------------- Orion / H1 (per-knockdown sums)
def cmd_sums(a, kind):
    genes = read_names(a.genes, "gene_name"); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    pc = label_pseudocount(a)
    with h5py.File(a.pseudobulk, "r") as f:
        obs = read_elem(f["obs"]); var = read_elem(f["var"])
        control = a.control
        if control is None:
            control = str(read_elem(f["uns/control_label"])) if "uns/control_label" in f else CONTROL
        names = names_of(obs); n = obs["n_cells"].values.astype(np.float64)
        src = alignment(var.index.astype(str), genes); ok = src >= 0
        if "measured" in var.columns:
            meas = np.zeros(len(genes), bool); meas[ok] = var["measured"].values.astype(bool)[src[ok]]
        else:
            meas = ok.copy()
        rc, r1, r2, re = (reader(f, f"layers/{k}") for k in ("sum_counts", "sum_cpm", "sum_cpm2", "n_expr"))
        controls = np.flatnonzero(names == control)
        if len(controls) != 1:
            raise ValueError("sums input needs one aggregated control row")
        ci = int(controls[0])

        def aligned(block):
            outb = np.full((block.shape[0], len(genes)), np.nan); outb[:, ok] = block[:, src[ok]]; return outb
        c_cnt, c_cpm, c_cpm2, c_det = (aligned(r([ci]))[0] for r in (rc, r1, r2, re))
        nc = n[ci]
        c_pool = c_cnt / np.nansum(np.where(meas, c_cnt, 0)) * 1e6
        c_mean = c_cpm / nc; c_var = np.maximum(c_cpm2 / nc - c_mean ** 2, 0)
        c_unit = 1e6 / max(float(np.nansum(np.where(meas, c_cnt, 0))), 1.0)
        c_serel2 = mean_variance(c_mean, c_var / nc, c_unit) / (c_mean + pc) ** 2
        keep = np.flatnonzero((names != control) & (n >= a.min_cells))
        # Check EACH guide/row before selecting among successful guides.
        keep, qc = filter_sum_guides(keep, names, rc, r1, aligned, n, genes,
                                    c_pool, c_mean, meas, a)
        pd.DataFrame(qc).to_csv(out / "guide_qc.csv", index=False)
        targets = names[keep]
        # duplicate names (e.g. several guides): keep the row with most cells
        order = np.lexsort((-n[keep], targets)); first = np.r_[True, targets[order][1:] != targets[order][:-1]]
        keep = np.sort(keep[order][first]); targets = names[keep]
        print(f"{kind}: {len(keep):,} knockdowns with >= {a.min_cells} cells; control '{control}' with {int(nc):,} cells; "
              f"{int(meas.sum()):,}/{len(genes):,} genes measured")

        def rows():
            for b in range(0, len(keep), a.chunk):
                r = keep[b:b + a.chunk]
                cnt, cpm, cpm2 = (aligned(x(r)) for x in (rc, r1, r2))
                nk = n[r][:, None]
                pool = cnt / np.maximum(np.nansum(np.where(meas, cnt, 0), 1, keepdims=True), 1) * 1e6
                mean = cpm / nk
                effect_base = pool if a.lfc_estimator == "pooled" else mean
                control_base = c_pool if a.lfc_estimator == "pooled" else c_mean
                lfc = np.log2((effect_base + pc) / (control_base + pc))
                mean = cpm / nk; var_ = np.maximum(cpm2 / nk - mean ** 2, 0)
                unit = 1e6 / np.maximum(np.nansum(np.where(meas, cnt, 0), 1, keepdims=True), 1)
                se = np.sqrt(mean_variance(mean, var_ / nk, unit) / (mean + pc) ** 2 + c_serel2) / np.log(2)
                lfc[:, ~meas] = np.nan; se[:, ~meas] = np.nan
                for k in range(len(r)):
                    yield b + k, lfc[k], se[k]
                print(f"  {min(b + a.chunk, len(keep)):,}/{len(keep):,}", end="\r", flush=True)
        if not len(targets): raise ValueError("no successful target groups after guide QC")
        write_effects(out, genes, targets, rows(), dict(
            kind=kind, source=str(Path(a.pseudobulk).resolve()), control_label=control, min_cells=a.min_cells,
            lfc=f"log2 ratio of {a.lfc_estimator} CPM, pseudocount {pc} CPM", label_pseudocount=pc,
            label_estimator=a.lfc_estimator, baseline_convention="ratio_of_sums_cpm",
            estimator_matches_official=False, guide_qc="drop measured on-target LFC >= threshold before choosing largest group",
            se_kind="cell_level (delta method from per-cell CPM spread, Poisson one-count floor; overconfident vs replicate SE)", legacy=False),
            len(keep))
    prof = dict(mean_cpm=np.where(meas, c_mean, np.nan).astype(np.float32),
                pooled_cpm=np.where(meas, c_pool, np.nan).astype(np.float32),
                detection=np.where(meas, c_det / nc, np.nan).astype(np.float32),
                var_cpm=np.where(meas, c_var, np.nan).astype(np.float32),
                se_mean=np.where(meas, np.sqrt(c_var / nc), np.nan).astype(np.float32),
                measured=meas, n_cells=np.int64(nc), legacy_baseline=np.bool_(False))
    prof["log_mean"] = np.log1p(prof["pooled_cpm"])
    save_profile(out / "control_profile.npz", genes, prof, dict(source=str(a.pseudobulk), kind=kind, baseline="controls row"))
    context_file(out, a.name, a.background, a.platform, a.priority, kind)
    print(f"\nwrote {out}  (lfc.npy, se.npy, control_profile.npz, context.json)")


# ---------------------------------------------------------------- T cells (already an effect table)
def cmd_tcell(a):
    genes = read_names(a.genes, "gene_name"); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    with h5py.File(a.effects, "r") as f:
        obs = read_elem(f["obs"]); var = read_elem(f["var"])
        names = names_of(obs); src = alignment(var.index.astype(str), genes); ok = src >= 0
        meas = np.zeros(len(genes), bool)
        meas[ok] = var["measured"].values.astype(bool)[src[ok]] if "measured" in var.columns else True
        ctrl = np.full(len(genes), np.nan); ctrl[ok] = var["ctrl_cpm"].values.astype(float)[src[ok]]
        rx = reader(f, "X"); rs = reader(f, "layers/se") if "layers/se" in f else None
        ncell = obs["n_cells"].values if "n_cells" in obs.columns else np.ones(len(names))
        order = np.lexsort((-ncell, names)); first = np.r_[True, names[order][1:] != names[order][:-1]]
        candidates = np.flatnonzero(names != CONTROL)
        vpos = {str(g): i for i, g in enumerate(var.index)}
        own_parts = []
        for start in range(0, len(candidates), a.chunk):
            part = candidates[start:start+a.chunk]; values = rx(part)
            own_parts.extend([values[i, vpos[t]] if t in vpos else np.nan for i, t in enumerate(names[part])])
        own = np.asarray(own_parts)
        passed = ~np.isfinite(own) | (own < a.min_ontarget_lfc)
        pd.DataFrame(dict(row=candidates, target=names[candidates], ontarget_lfc=own,
                          passed=passed, measured=np.isfinite(own))).to_csv(out / "guide_qc.csv", index=False)
        candidates = candidates[passed]
        order = np.lexsort((-ncell[candidates], names[candidates]))
        ordered = candidates[order]
        first = np.r_[True, names[ordered][1:] != names[ordered][:-1]] if len(ordered) else np.array([], bool)
        keep = np.sort(ordered[first]); targets = names[keep]
        print(f"tcell: {len(keep):,} knockdowns, {int(meas.sum()):,}/{len(genes):,} genes measured, SE: {rs is not None}")

        def rows():
            for b in range(0, len(keep), a.chunk):
                r = keep[b:b + a.chunk]
                L = np.full((len(r), len(genes)), np.nan); S = np.full_like(L, np.nan)
                L[:, ok] = rx(r)[:, src[ok]]
                if rs is not None:
                    S[:, ok] = rs(r)[:, src[ok]]
                L[:, ~meas] = np.nan; S[:, ~meas] = np.nan
                for k in range(len(r)):
                    yield b + k, L[k], S[k]
                print(f"  {min(b + a.chunk, len(keep)):,}/{len(keep):,}", end="\r", flush=True)
        if not len(targets): raise ValueError("no successful target groups after guide QC")
        write_effects(out, genes, targets, rows(), dict(
            kind="tcell", source=str(Path(a.effects).resolve()),
            lfc="replicate-paired log2 ratio of pseudobulk CPM (tcell_build_effects.py)",
            se_kind="donor_level (limma-style moderated)", legacy=rs is None), len(keep))
    pooled = np.where(meas, ctrl, np.nan).astype(np.float32)
    prof = dict(mean_cpm=pooled, pooled_cpm=pooled, log_mean=np.log1p(pooled),
                detection=np.full(len(genes), np.nan, np.float32), var_cpm=np.full(len(genes), np.nan, np.float32),
                se_mean=np.full(len(genes), np.nan, np.float32), measured=meas, n_cells=np.int64(0),
                legacy_baseline=np.bool_(True))     # baseline from pseudobulk, not single cells
    save_profile(out / "control_profile.npz", genes, prof, dict(source=str(a.effects), kind="tcell",
                 baseline="var['ctrl_cpm'] (pseudobulk; mean_cpm == pooled_cpm)"))
    context_file(out, a.name, a.background, a.platform, a.priority, "tcell")
    print(f"\nwrote {out}")


def filter_sum_guides(keep, names, rc, r1, aligned, n, genes, c_pool, c_mean, meas, a):
    gp = {g: i for i, g in enumerate(genes)}; retained, qc = [], []; pc = label_pseudocount(a)
    for start in range(0, len(keep), a.chunk):
        r = keep[start:start+a.chunk]
        cnt, cpm = aligned(rc(r)), aligned(r1(r))
        pool = cnt/np.maximum(np.nansum(np.where(meas, cnt, 0), axis=1, keepdims=True), 1)*1e6
        mean = cpm/n[r, None]
        base, ctrl = (pool, c_pool) if a.lfc_estimator == "pooled" else (mean, c_mean)
        for i, row in enumerate(r):
            j = gp.get(names[row])
            own = np.log2((base[i, j]+pc)/(ctrl[j]+pc)) if j is not None and meas[j] else np.nan
            passed = not np.isfinite(own) or own < a.min_ontarget_lfc
            qc.append(dict(row=int(row), target=names[row], ontarget_lfc=float(own),
                           measured=bool(np.isfinite(own)), passed=bool(passed)))
            if passed: retained.append(row)
    return np.asarray(retained, int), qc


def cmd_cells(a):
    """Raw Replogle/H1/etc. cells: guide QC, official 2026 arithmetic CPM labels.

    Normalization is per cell to 1e6 on the aligned gene axis, matching
    cell-eval2 vcc2026's arithmetic CPM convention. Labels add LABEL_PSEUDOCOUNT CPM
    (not the evaluator's 1e-9) so unscored low-count genes cannot produce huge labels.
    Delta-method SE remains cell-level and is identified as such in metadata.
    """
    genes = read_names(a.genes, "gene_name"); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    pc = label_pseudocount(a)
    h = ad.read_h5ad(a.cells, backed="r")
    try:
        source_genes = h.var[a.gene_col].astype(str).values if a.gene_col else h.var_names.astype(str)
        if len(set(source_genes)) != len(source_genes):
            raise ValueError("raw source gene symbols must be unique; aggregate duplicate symbols first")
        idx = alignment(source_genes, genes); ok = idx >= 0
        labels = h.obs[a.obs_col].astype(str).values
        if a.target_split:
            labels = np.array([x.split(a.target_split)[0] for x in labels])
        control_values = a.control.split(',')
        ci = np.flatnonzero(np.isin(labels, control_values))
        if not len(ci): raise ValueError("no controls; set --control to the source's exact control label")
        matrix = h.layers[a.layer] if a.layer else h.X
        libraries = []
        for start in range(0, h.n_obs, a.block):
            m = matrix[start:start+a.block]
            validate_counts(m, a.cells)
            libraries.append(np.asarray(m[:, idx[ok]].sum(1)).ravel())
        libraries = np.concatenate(libraries)
        positive = libraries[libraries > 0]
        if not len(positive): raise ValueError("no positive raw libraries")
        scale = 1e6

        def normalized_stats(rows):
            total = np.zeros(ok.sum()); squares = np.zeros(ok.sum())
            for start in range(0, len(rows), a.block):
                rr = rows[start:start+a.block]
                m = matrix[rr]
                m = (m.tocsr() if sp.issparse(m) else sp.csr_matrix(m))[:, idx[ok]].astype(float)
                m = sp.diags(scale/np.maximum(libraries[rr], 1)) @ m
                total += np.asarray(m.sum(0)).ravel(); squares += np.asarray(m.multiply(m).sum(0)).ravel()
            avg = total/len(rows); natural = avg
            var = np.maximum(squares/len(rows)-avg**2, 0)/len(rows)
            unit = float(np.mean(scale/np.maximum(libraries[rows], 1)))/len(rows)
            relvar = mean_variance(natural, var, unit)/(natural+pc)**2
            full = np.full(len(genes), np.nan); rv = full.copy()
            full[ok] = natural; rv[ok] = relvar
            return full, rv

        ctrl, ctrl_var = normalized_stats(ci)
        prof = moments(h, ci, idx, a.layer, a.block)
        save_profile(out / "control_profile.npz", genes, prof, dict(kind=a.kind, baseline_convention="ratio_of_sums_cpm"))
        gp = {g: i for i, g in enumerate(genes)}; qc = []; groups = {}
        guides = h.obs[a.guide_col].astype(str).values if a.guide_col else labels
        # A missing guide column means all cells of the target form one group.
        grouped = pd.DataFrame({'target': labels, 'guide': guides}).groupby(['target', 'guide'], sort=True).indices
        for (target, guide), rr in grouped.items():
            if target in control_values: continue
            if len(rr) < a.min_cells: continue
            mean, rv = normalized_stats(rr)
            lfc = np.log2((mean+pc)/(ctrl+pc))
            j = gp.get(target); own = lfc[j] if j is not None else np.nan
            passed = not np.isfinite(own) or own < a.min_ontarget_lfc
            qc.append(dict(target=target, guide=guide, n_cells=len(rr), ontarget_lfc=own,
                           measured=np.isfinite(own), passed=passed))
            if passed and (target not in groups or len(rr) > groups[target][0]):
                groups[target] = (len(rr), lfc, np.sqrt(rv+ctrl_var)/np.log(2))
        targets = sorted(groups)
        if not targets: raise ValueError("no successful target groups after guide QC")
        pd.DataFrame(qc).to_csv(out / "guide_qc.csv", index=False)
        write_effects(out, genes, targets, ((i, groups[t][1], groups[t][2]) for i, t in enumerate(targets)),
                      dict(kind=a.kind, source=str(Path(a.cells).resolve()), label_estimator="per_cell_cpm",
                           normalization_target=scale, label_pseudocount=pc, official_epsilon=PSEUDO,
                           estimator_matches_official=True,
                           baseline_convention="ratio_of_sums_cpm", se_kind="cell_level_delta_per_cell_cpm_poisson_floor",
                           legacy=False), len(targets))
        context_file(out, a.name, a.background, a.platform, a.priority, a.kind)
    finally:
        h.file.close()
    print(f"wrote {len(targets)} successful raw-cell targets to {out}")


def cmd_manifest(a):
    """Explicit raw-cell manifest avoids guessing source labels/layers or downloading huge inputs."""
    records = pd.read_csv(a.manifest).fillna('').to_dict('records')
    for record in records:
        required = ['cells', 'name', 'background', 'obs_col', 'control']
        if any(not record.get(k) for k in required): raise ValueError(f"manifest needs {required}")
        options = dict(cells=record['cells'], genes=a.genes, out=str(Path(a.out)/record['name']),
                       name=record['name'], background=record['background'], obs_col=record['obs_col'],
                       control=record['control'], platform=record.get('platform') or '3prime',
                       kind=record.get('kind') or 'replogle', guide_col=record.get('guide_col') or None,
                       gene_col=record.get('gene_col') or None, layer=record.get('layer') or None,
                       target_split=record.get('target_split') or None, priority=int(record.get('priority') or 0),
                       min_cells=20, min_ontarget_lfc=-0.1, block=512, pseudocount=LABEL_PSEUDOCOUNT)
        cmd_cells(argparse.Namespace(**options))


# ---------------------------------------------------------------- contexts.json
def cmd_config(a):
    contexts = []
    for directory in a.dirs:
        d = Path(directory); c = json.loads((d / "context.json").read_text())
        contexts.append(dict(name=c["name"], background=c["background"], platform=c["platform"],
                             source_priority=c.get("source_priority", 0), effects=str(d.resolve()),
                             profile=str((d / "control_profile.npz").resolve())))
    if len({c['background'] for c in contexts}) < 2:
        raise ValueError('need at least two biological backgrounds')
    if len({c['name'] for c in contexts}) != len(contexts): raise ValueError('duplicate context name')
    cfg = dict(schema='vcc_2026_2_average_v1', genes=str(Path(a.genes).resolve()), contexts=contexts,
               weighting='equal_background', baseline_mode='bulk', signal_z=2.0, expression_min_cpm=5.0,
               neighbours=str(Path(a.neighbours).resolve()) if a.neighbours else None, neighbour_k=5)
    save_json(a.out, cfg)
    print(json.dumps(cfg, indent=2))


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    for kind, bg, plat in (("orion", None, "orion"), ("h1", "H1", "flex")):
        s = sub.add_parser(kind); s.add_argument("--pseudobulk", required=True); s.add_argument("--genes", required=True)
        s.add_argument("--out", required=True); s.add_argument("--name", required=True)
        s.add_argument("--background", required=bg is None, default=bg); s.add_argument("--platform", default=plat)
        s.add_argument("--priority", type=int, default=0); s.add_argument("--control")
        s.add_argument("--min-ontarget-lfc", type=float, default=-0.1)
        s.add_argument("--lfc-estimator", choices=["per_cell", "pooled"], default="per_cell")
        s.add_argument("--min-cells", type=int, default=20); s.add_argument("--chunk", type=int, default=256)
        s.add_argument("--pseudocount", type=float, default=LABEL_PSEUDOCOUNT, help="CPM added to both sides of label ratios")
    t = sub.add_parser("tcell"); t.add_argument("--effects", required=True); t.add_argument("--genes", required=True)
    t.add_argument("--out", required=True); t.add_argument("--name", required=True)
    t.add_argument("--background", default="CD4_T"); t.add_argument("--platform", default="flex")
    t.add_argument("--priority", type=int, default=0); t.add_argument("--chunk", type=int, default=256)
    t.add_argument("--min-ontarget-lfc", type=float, default=-0.1)
    raw = sub.add_parser("cells")
    for key in ["cells", "genes", "out", "name", "background", "obs-col", "control"]:
        raw.add_argument("--"+key, required=True)
    for key in ["guide-col", "gene-col", "layer", "target-split"]: raw.add_argument("--"+key)
    raw.add_argument("--kind", default="replogle"); raw.add_argument("--platform", default="3prime")
    raw.add_argument("--priority", type=int, default=0); raw.add_argument("--min-cells", type=int, default=20)
    raw.add_argument("--min-ontarget-lfc", type=float, default=-0.1)
    raw.add_argument("--block", type=int, default=512)
    raw.add_argument("--pseudocount", type=float, default=LABEL_PSEUDOCOUNT, help="CPM added to both sides of label ratios")
    manifest = sub.add_parser("manifest")
    for key in ["manifest", "genes", "out"]: manifest.add_argument("--"+key, required=True)
    m = sub.add_parser("make-config"); m.add_argument("--dirs", nargs="+", required=True)
    m.add_argument("--genes", required=True); m.add_argument("--out", required=True)
    m.add_argument("--neighbours", help="CSV gene_a,gene_b,score from coessentiality/STRING/CORUM")
    a = ap.parse_args()
    {"orion": lambda: cmd_sums(a, "orion"), "h1": lambda: cmd_sums(a, "h1"),
     "cells": lambda: cmd_cells(a), "manifest": lambda: cmd_manifest(a),
     "tcell": lambda: cmd_tcell(a), "make-config": lambda: cmd_config(a)}[a.cmd]()


### vcc2_context_profile.py


In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_context_profile.py
"""Profile an unseen line using ALL available raw-count control cells.

Transfer context uses ratio-of-sums CPM (bulk_cpm), not mean per-cell CPM.
Both are retained: mean_cpm/var_cpm/se_mean keep their original per-cell meaning
for DE diagnostics and the existing cell generator. This does not convert LFCs.

Example: python vcc2_context_profile.py --counts context_A.h5ad --all-controls
         --genes gene_names.csv --out profiles/A.npz
"""
import argparse
from pathlib import Path

import anndata as ad
import numpy as np

from vcc2_common import alignment, moments, program_levels, read_names, save_profile, select_controls


def profile(counts, genes_csv, output, layer=None, obs_col="target_gene",
            control="non-targeting", all_controls=False, block=512):
    genes = read_names(genes_csv, "gene_name")
    a = ad.read_h5ad(counts, backed="r")
    try:
        idx = alignment(a.var_names.astype(str), genes)
        rows = select_controls(a, obs_col, control, all_controls)
        stats = moments(a, rows, idx, layer, block)
        # Explicit transfer representation. Do not overwrite mean_cpm/log_mean:
        # the DE scorer and the cell generator still need per-cell mean CPM.
        stats["baseline_cpm"] = stats["pooled_cpm"].copy()
        stats["baseline_log"] = np.log1p(stats["pooled_cpm"])
        stats["baseline_convention"] = np.array("ratio_of_sums_cpm")
        stats["program_names"], stats["baseline_program_levels"] = program_levels(stats["pooled_cpm"], genes)
        finite = np.isfinite(stats["baseline_log"])
        centre = float(stats["baseline_log"][finite].mean())
        stats["baseline_log_centered"] = stats["baseline_log"] - centre
        # These identifiers permit auditing overlap when making synthetic predictions.
        stats["control_obs_names"] = np.asarray(a.obs_names[rows], dtype=str)
        save_profile(output, genes, stats, dict(counts=str(Path(counts).resolve()), layer=layer,
                     obs_col=obs_col, control=control, all_controls=all_controls,
                     transfer_baseline_convention="ratio_of_sums_cpm",
                     transfer_baseline_field="pooled_cpm",
                     per_cell_mean_field="mean_cpm",
                     baseline_log_center=centre,
                     library_denominator="sum over the provided gene_names.csv axis"))
    finally:
        a.file.close()
    print(f"Saved {output}: {len(rows):,} controls, {len(genes):,} aligned genes")


if __name__ == "__main__":
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--counts", required=True)
    p.add_argument("--genes", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--layer")
    p.add_argument("--obs-col", default="target_gene")
    p.add_argument("--control", default="non-targeting")
    p.add_argument("--all-controls", action="store_true")
    p.add_argument("--block", type=int, default=512)
    a = p.parse_args()
    profile(a.counts, a.genes, a.out, a.layer, a.obs_col, a.control, a.all_controls, a.block)


### vcc2_transfer.py


In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_transfer.py
"""Fixed averaging of once-filtered source effects, one source per background.

Context profiles retain their existing pooled-CPM representation. They are
reported for inspection; equal background weights are the baseline. Expression
gating happens once in count generation, using mean per-cell CPM of real controls.
There is no prior, fitted source scale, ridge, or transfer-parameter search.
"""
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
from vcc2_common import (Effects, load_profile, program_levels, read_names,
                         save_json, seed_for, significance_filter)

SCHEMA = 'vcc_2026_2_average_v1'


def transfer_profile(profile, mode='bulk', declared_convention=None, label='profile'):
    p = dict(profile)
    convention = declared_convention or str(p.get('baseline_convention', ''))
    if mode != 'bulk':
        raise ValueError('this baseline preserves the existing pooled-CPM context profiles')
    if 'pooled_cpm' in p: base = p['pooled_cpm']
    elif 'bulk_cpm' in p: base = p['bulk_cpm']
    elif convention == 'ratio_of_sums_cpm': base = p['mean_cpm']
    else: raise ValueError(f'{label}: no verified pooled CPM baseline')
    base = np.asarray(base, float)
    if not np.isfinite(base).any() or np.any(base[np.isfinite(base)] < 0) or np.isinf(base).any():
        raise ValueError(f'{label}: invalid baseline')
    p['baseline_cpm'] = base
    p['baseline_log'] = np.log1p(base)
    _, p['baseline_program_levels'] = program_levels(base, p['genes'])
    return p


def centred_similarity(a, b):
    valid = np.isfinite(a) & np.isfinite(b)
    if valid.sum() < 2: return 0.0
    x, y = np.asarray(a[valid], float).copy(), np.asarray(b[valid], float).copy()
    x -= x.mean(); y -= y.mean()
    denominator = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.clip(np.dot(x, y) / denominator, -1, 1)) if denominator > 1e-12 else 0.0


class Context:
    def __init__(self, spec, genes, allow_legacy=False):
        self.name = spec['name']; self.background = spec['background']
        self.priority = spec.get('source_priority', 0); self.spec = spec
        self.effects = Effects(spec['effects'], genes, allow_legacy)
        self.profile = transfer_profile(load_profile(spec.get('profile', str(Path(spec['effects']) / 'control_profile.npz')), genes),
                                        declared_convention=spec.get('baseline_convention'), label=self.name)


def load_contexts(config, genes, excluded=(), allow_legacy=False):
    # Exclusion happens before opening ANY effects or profiles of the held-out line.
    return [Context(c, genes, allow_legacy) for c in config['contexts'] if c['background'] not in excluded]


def choose_sources(contexts, target_background=None):
    # Apply to candidates that measured this knockdown; do not let duplicate
    # releases or two CD4_T states count as independent biological backgrounds.
    groups = {}
    for c in contexts:
        if c.background == target_background: continue
        if c.background not in groups or (c.priority, c.name) > (groups[c.background].priority, groups[c.background].name):
            groups[c.background] = c
    return sorted(groups.values(), key=lambda c: c.name)


class FunctionalNeighbours:
    def __init__(self, path=None, k=5):
        if k < 1: raise ValueError('neighbour_k must be positive')
        self.k, self.adj = k, {}
        if path:
            frame = pd.read_csv(path, sep=None, engine='python')
            if not {'gene_a', 'gene_b', 'score'} <= set(frame):
                raise ValueError('neighbours need gene_a,gene_b,score columns')
            for a, b, score in frame[['gene_a', 'gene_b', 'score']].itertuples(index=False, name=None):
                if not np.isfinite(score) or score <= 0 or a == b: continue
                for x, y in [(str(a), str(b)), (str(b), str(a))]:
                    self.adj.setdefault(x, {})[y] = max(float(score), self.adj.get(x, {}).get(y, 0))

    def available(self, target, covered):
        return sorted([(g, s) for g, s in self.adj.get(target, {}).items() if g in covered],
                      key=lambda pair: (-pair[1], pair[0]))[:self.k]


class FilteredEffects:
    """Filter each measured donor once, then optionally combine filtered neighbours.

    Insignificant measured effects are zeros and remain in averaging denominators.
    Missing/invalid uncertainty is unknown (NaN), never a fabricated zero or SE.
    Neighbour combinations are not significance-filtered again.
    """
    def __init__(self, original, graph, genes):
        self.original, self.graph = original, graph
        self.pos, self.targets = original.pos, original.targets
        self.gene_pos = {str(g): i for i, g in enumerate(genes)}

    def neighbours(self, kd): return self.graph.available(kd, self.pos)

    def get(self, kd, gi):
        if kd in self.pos:
            return significance_filter(*self.original.get(kd, gi))
        neighbours = self.neighbours(kd)
        if not neighbours: return np.full(len(gi), np.nan, np.float32)
        values = np.stack([significance_filter(*self.original.get(g, gi)) for g, _ in neighbours])
        columns = np.asarray(gi)
        for row, (g, _) in enumerate(neighbours):
            # Never mistake a neighbour's own knockdown for a downstream response.
            own = self.gene_pos.get(g)
            if own is not None: values[row, columns == own] = np.nan
        weight = np.asarray([s for _, s in neighbours])[:, None] * np.isfinite(values)
        den = weight.sum(0)
        return np.divide((np.nan_to_num(values) * weight).sum(0), den,
                         out=np.full(len(gi), np.nan), where=den > 0).astype(np.float32)


def average_effects(sources, kd, gi):
    """Equal biological-background average; missing observations never dilute it."""
    if not sources: return np.zeros(len(gi), np.float32), np.zeros(len(gi), np.uint16)
    values = np.stack([c.effects.get(kd, gi) for c in sources])
    number = np.isfinite(values).sum(0)
    result = np.divide(np.nan_to_num(values).sum(0), number,
                       out=np.zeros(len(gi)), where=number > 0)
    return result.astype(np.float32), number.astype(np.uint16)


def prepare(config_path, output, holdout=None):
    """Register fixed averaging inputs; no fitted coefficients or tuning folds."""
    cfg = json.loads(Path(config_path).read_text()); genes = read_names(cfg['genes'], 'gene_name')
    if cfg.get('schema') != SCHEMA: raise ValueError('rebuild contexts.json with this notebook; old settings are incompatible')
    if holdout and holdout not in {c['background'] for c in cfg['contexts']}:
        raise ValueError('unknown holdout background')
    contexts = load_contexts(cfg, genes, [holdout] if holdout else [])
    if not contexts: raise ValueError('no source backgrounds remain')
    artifact = dict(schema=SCHEMA, genes=genes.tolist(), contexts=[c.spec for c in contexts],
                    excluded_background=holdout, neighbours=cfg.get('neighbours'),
                    neighbour_k=int(cfg.get('neighbour_k', 5)), selected_method='source_mean',
                    weighting='equal_background', signal_filter='abs(lfc) > 2 * SE, applied to measured donors once',
                    missing_se='exclude; unknown is not a measured null', expression_gate='mean per-cell CPM > 5; count generator only',
                    context_profile='existing log1p pooled CPM; similarity is diagnostic only')
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    save_json(out / 'model.json', artifact)
    save_json(out / 'model_report.json', dict(artifact, source_backgrounds=sorted({c.background for c in contexts}),
                                            fitted_parameters=0, labels={c.name: c.effects.meta for c in contexts}))
    print(f'Prepared fixed average: {len(contexts)} tables; excluded background={holdout}')
    return artifact


def infer(model_dir, profile_path, targets_csv, output, chunk=2048, allow_unsupported=False):
    artifact = json.loads((Path(model_dir) / 'model.json').read_text())
    if artifact.get('schema') != SCHEMA: raise ValueError('incompatible model; prepare with this notebook')
    if chunk < 1: raise ValueError('chunk must be positive')
    genes = np.asarray(artifact['genes']); targets = read_names(targets_csv, 'target_gene')
    if not len(targets): raise ValueError('empty perturbation panel')
    profile = transfer_profile(load_profile(profile_path, genes))
    contexts = load_contexts(artifact, genes)
    graph = FunctionalNeighbours(artifact.get('neighbours'), artifact['neighbour_k'])
    for c in contexts: c.effects = FilteredEffects(c.effects, graph, genes)
    usable_for = {}
    for kd in targets:
        direct = [c for c in contexts if kd in c.effects.pos]
        usable_for[kd] = choose_sources(direct or [c for c in contexts if c.effects.neighbours(kd)])
    unresolved = [kd for kd in targets if not usable_for[kd]]
    if unresolved and not allow_unsupported:
        raise ValueError(f'unsupported targets: {unresolved[:20]}; add measured sources/neighbours, or explicitly enable --allow-unsupported for reported zeros')
    out = Path(output)
    if out.exists() and any(out.iterdir()): raise ValueError('choose an empty prediction directory')
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / 'genes.npy', genes); np.save(out / 'targets.npy', targets)
    from numpy.lib.format import open_memmap
    arrays = {name: open_memmap(out / f'{name}.npy', mode='w+', dtype='float32', shape=(len(targets), len(genes)))
              for name in ['source_mean', 'selected']}
    counts = open_memmap(out / 'source_count.npy', mode='w+', dtype='uint16', shape=(len(targets), len(genes)))
    own = np.full(len(targets), np.nan, np.float32); gp = {g: i for i, g in enumerate(genes)}
    diagnostics, no_evidence = [], []
    similarities = {c.name: centred_similarity(profile['baseline_log'], c.profile['baseline_log']) for c in contexts}
    for row, kd in enumerate(targets):
        sources = usable_for[kd]; covered = 0
        for start in range(0, len(genes), chunk):
            gi = np.arange(start, min(start + chunk, len(genes)))
            mean, number = average_effects(sources, kd, gi)
            for arr in arrays.values(): arr[row, gi] = mean
            counts[row, gi] = number; covered += int((number > 0).sum())
        if not covered: no_evidence.append(kd)
        ki = gp.get(kd)
        if ki is not None:
            # Guide efficiency is separate from downstream DE: use raw measured,
            # successful negative on-target effects, with one vote per background.
            measured = [c.effects.original.get(kd, [ki])[0][0] for c in sources if kd in c.effects.pos]
            measured = [v for v in measured if np.isfinite(v) and v < -0.1]
            if measured: own[row] = np.mean(measured)
        diagnostics.append(dict(target=kd, covered_genes=covered, source_names=[c.name for c in sources],
                                background_weights={c.background: 1 / len(sources) for c in sources},
                                borrowed={c.name: c.effects.neighbours(kd) for c in sources if kd not in c.effects.pos}))
    for arr in arrays.values(): arr.flush()
    counts.flush(); np.save(out / 'ontarget_lfc.npy', own)
    save_json(out / 'prediction_report.json', dict(schema=SCHEMA, selected_method='source_mean',
              heldout_background=artifact['excluded_background'], source_backgrounds=sorted({c.background for c in contexts}),
              model_dir=str(Path(model_dir).resolve()), target_profile=str(Path(profile_path).resolve()),
              source_names=[c.name for c in contexts], context_similarities=similarities, weighting='equal_background',
              coverage=diagnostics, unresolved=unresolved, no_usable_uncertainty=no_evidence,
              signal_filter=artifact['signal_filter'], expression_gate=artifact['expression_gate'],
              allow_unsupported=allow_unsupported))
    if no_evidence and not allow_unsupported:
        raise ValueError(f'no usable uncertainty for targets {no_evidence[:20]}; see prediction_report.json before permitting zeros')


def holdout_panels(config_path, output, max_targets=1000, seed=2026):
    cfg = json.loads(Path(config_path).read_text())
    measured = {c['name']: set(np.load(Path(c['effects']) / 'targets.npy').astype(str)) for c in cfg['contexts']}
    out = Path(output); out.mkdir(parents=True, exist_ok=True); summary = {}
    for c in cfg['contexts']:
        others = set().union(*(measured[o['name']] for o in cfg['contexts'] if o['background'] != c['background']))
        covered = sorted(measured[c['name']] & others); total = len(covered)
        if max_targets and total > max_targets:
            covered = sorted(np.random.default_rng(seed_for(seed, 'panel', c['name'])).choice(covered, max_targets, replace=False).tolist())
        pd.DataFrame({'target_gene': covered}).to_csv(out / f"{c['name']}.csv", index=False)
        summary[c['name']] = dict(background=c['background'], covered_by_other_backgrounds=total, panel=len(covered))
    save_json(out / 'panels_summary.json', dict(seed=seed, max_targets=max_targets, contexts=summary))
    return summary


def coverage(config_path, targets_path, output, profiles=None):
    cfg = json.loads(Path(config_path).read_text()); targets = read_names(targets_path, 'target_gene')
    graph = FunctionalNeighbours(cfg.get('neighbours'), int(cfg.get('neighbour_k', 5)))
    sources = {c['name']: set(np.load(Path(c['effects']) / 'targets.npy').astype(str)) for c in cfg['contexts']}
    union = set().union(*sources.values()); records = []
    for kd in targets:
        direct = [name for name, known in sources.items() if kd in known]
        neighbours = graph.available(kd, union)
        records.append(dict(target_gene=kd, direct=bool(direct), sources=';'.join(direct),
                            borrowed=bool(not direct and neighbours), neighbours=';'.join(g for g, _ in neighbours)))
    frame = pd.DataFrame(records); out = Path(output); out.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out / 'coverage.csv', index=False)
    summary = dict(n_targets=len(frame), direct_fraction=float(frame.direct.mean()),
                   resolved_fraction=float((frame.direct | frame.borrowed).mean()),
                   unresolved=frame.loc[~frame.direct & ~frame.borrowed, 'target_gene'].tolist(),
                   note='target availability after guide QC; prediction reports also count usable gene-level uncertainty')
    if profiles:
        genes = read_names(cfg['genes'], 'gene_name'); contexts = load_contexts(cfg, genes)
        summary['lines'] = {}
        for spec in profiles:
            line, path = spec.split('=', 1); p = transfer_profile(load_profile(path, genes))
            summary['lines'][line] = dict(genes_above_5_cpm=int((p['mean_cpm'] > 5).sum()),
                context_similarities={c.name: centred_similarity(p['baseline_log'], c.profile['baseline_log']) for c in contexts})
    save_json(out / 'coverage_summary.json', summary); print(json.dumps(summary, indent=2))


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest='cmd', required=True)
    prep = sub.add_parser('prepare')
    for k in ['config', 'out']: prep.add_argument('--' + k, required=True)
    prep.add_argument('--holdout')
    pred = sub.add_parser('predict')
    for k in ['model', 'profile', 'targets', 'out']: pred.add_argument('--' + k, required=True)
    pred.add_argument('--chunk', type=int, default=2048); pred.add_argument('--allow-unsupported', action='store_true')
    cov = sub.add_parser('coverage')
    for k in ['config', 'targets', 'out']: cov.add_argument('--' + k, required=True)
    cov.add_argument('--profile', action='append')
    panels = sub.add_parser('panels')
    for k in ['config', 'out']: panels.add_argument('--' + k, required=True)
    panels.add_argument('--max-targets', type=int, default=1000); panels.add_argument('--seed', type=int, default=2026)
    a = p.parse_args()
    if a.cmd == 'prepare': prepare(a.config, a.out, a.holdout)
    elif a.cmd == 'predict': infer(a.model, a.profile, a.targets, a.out, a.chunk, a.allow_unsupported)
    elif a.cmd == 'coverage': coverage(a.config, a.targets, a.out, a.profile)
    else: holdout_panels(a.config, a.out, a.max_targets, a.seed)


### vcc2_generate.py


In [ ]:
%%writefile ~/vcc/code/vcc_2026_2/vcc2_generate.py
"""Generate counts with projected composition, metadata cell counts/library sizes,
state-dependent overdispersed induction and reproducible common control sampling.

`benchmark` tunes on LOBO-H1 transfer predictions with official cell-eval2 (vcc2026), streaming every
file one target at a time; the real side is cached once and the ceiling computed once.
`oracle` is a separate generator ceiling check and never selects submission knobs.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import shlex
import subprocess
import shutil
import copy
from vcc2_common import int64_indptr, seed_for, validate_counts, save_json, expression_gate, moments, alignment

try:
    from anndata.io import read_elem, write_elem, sparse_dataset
except ImportError:  # older anndata
    from anndata.experimental import read_elem, write_elem, sparse_dataset

VCC2026_METRICS = {
    'pds_cosine': 'max', 'expr_mse_unbiased_capped_norm': 'min',
    'de_wilcoxon_lfc_nmae': 'min', 'de_wilcoxon_direction_fidelity_yield_raw': 'max',
    'de_wilcoxon_direction_reach_raw': 'max', 'de_wilcoxon_sig_jaccard': 'max',
}
CONTROL = "non-targeting"
CC_GENES = ("MCM5 PCNA TYMS FEN1 MCM2 MCM4 RRM1 UNG GINS2 MCM6 CDCA7 DTL PRIM1 UHRF1 HELLS RFC2 RPA2 NASP RAD51AP1 GMNN "
            "WDR76 SLBP CCNE2 UBR7 POLD3 MSH2 ATAD2 RAD51 RRM2 CDC45 CDC6 EXO1 TIPIN DSCC1 BLM CASP8AP2 USP1 CLSPN POLA1 "
            "CHAF1B BRIP1 E2F8 HMGB2 CDK1 NUSAP1 UBE2C BIRC5 TPX2 TOP2A NDC80 CKS2 NUF2 CKS1B MKI67 TMPO CENPF TACC3 SMC4 "
            "CCNB2 CKAP2L CKAP2 AURKB BUB1 KIF11 ANP32E TUBB4B GTSE1 KIF20B HJURP CDCA3 CDC20 TTK CDC25C KIF2C RANGAP1 "
            "NCAPD2 DLGAP5 CDCA2 CDCA8 ECT2 KIF23 HMMR AURKA PSRC1 ANLN LBR CKAP5 CENPE CTCF NEK2 G2E3 GAS2L3 CBX5 CENPA").split()


class Cfg:
    def __init__(self, a):
        self.alpha = getattr(a, "alpha", 1.0); self.max_lfc = 8.0; self.min_lfc = 0.0
        self.kd_clip = (0.01, 0.9); self.default_kd = 0.30; self.n_cells = getattr(a, "n_cells", 400)
        self.composition = "projection"; self.comp_thr = getattr(a, "comp_thr", 0.02)
        self.pi = getattr(a, "pi", 0.0); self.up_w = getattr(a, "up_w", 0.75)
        self.tilt = getattr(a, "tilt", False); self.match_pool = getattr(a, "match_pool", False) or self.tilt
        self.seed = getattr(a, "seed", 2026)
        self.metadata = getattr(a, "metadata", True)
        self.library_size = None
        if not 0 <= self.pi < 1 or not 0 <= self.up_w <= 1 or self.n_cells < 1 or self.alpha < 0:
            raise ValueError("invalid generator settings")

    def as_dict(self):
        return {k: v for k, v in vars(self).items()}


# ---------------------------------------------------------------- inputs
def read_list(path, col):
    return pd.read_csv(path)[col].astype(str).tolist()


def aligned_X(a, genes):
    """cells x our genes, CSR float32 (missing genes = 0, extra genes dropped)."""
    X = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
    names = a.var_names.astype(str).tolist()
    if names == list(genes):
        return X.astype(np.float32)
    pos = {g: i for i, g in enumerate(genes)}
    src = np.array([i for i, g in enumerate(names) if g in pos]); dst = np.array([pos[names[i]] for i in src])
    X = X[:, src].tocoo()
    print(f"  re-indexed genes: {len(src):,}/{len(genes):,} of ours present")
    return sp.csr_matrix((X.data.astype(np.float32), (X.row, dst[X.col])), shape=(X.shape[0], len(genes)))


class Line:
    """Control cells of one context plus the statistics the generator needs."""
    def __init__(self, X, guide, genes):
        validate_counts(X, "control cells")
        self.X = X.tocsr().astype(np.float32); self.genes = genes; self.n = X.shape[0]
        if not self.n or self.X.sum() <= 0: raise ValueError("controls must contain cells and positive counts")
        self.lib = np.asarray(self.X.sum(1)).ravel()
        self.cpm_cells = sp.diags(1e6 / np.maximum(self.lib.astype(np.float64), 1)) @ self.X
        self.cpm = np.asarray(self.cpm_cells.mean(0)).ravel()           # mean per-cell CPM (scorer gate uses > 5)
        self.share = np.asarray(self.X.sum(0)).ravel() / self.X.sum()   # pooled share of molecules
        self.gate = expression_gate(self.cpm)
        count_mean = np.asarray(self.X.mean(0)).ravel()
        count_var = np.maximum(np.asarray(self.X.multiply(self.X).mean(0)).ravel()-count_mean**2, 0)
        self.overdispersion = np.where(count_mean > 0, np.clip((count_var-count_mean)/np.maximum(count_mean**2, 1e-12), 0, 10), 1.0)
        g = np.asarray(guide).astype(str) if guide is not None else np.zeros(self.n, str)
        self.groups = [np.flatnonzero(g == u) for u in np.unique(g)]
        gp = {x: i for i, x in enumerate(genes)}
        self.cc_idx = np.array([gp[x] for x in CC_GENES if x in gp and self.cpm[gp[x]] > 1], int)
        if len(self.cc_idx) >= 5:
            L = self.cpm_cells[:, self.cc_idx].toarray()
            self.cyc = np.log2(L + 1).mean(1)                            # per-cell cycling level (log2 units)
        else:
            self.cyc = None

    @staticmethod
    def from_h5ad(path, genes, control_col="target_gene"):
        a = ad.read_h5ad(path)
        validate_counts(a.X, str(path))
        if control_col and control_col in a.obs:
            a = a[a.obs[control_col].astype(str).values == CONTROL].copy()
        guide = next((a.obs[c].values for c in ("ntc_id", "guide_id", "sgID_AB", "guide") if c in a.obs), None)
        return Line(aligned_X(a, genes), guide, genes)


def load_deltas(path, targets, genes, array="source_mean"):
    """averaging prediction dir (genes.npy, targets.npy, <array>.npy) or .npz (delta [+targets, genes]) -> T x G, NaN -> 0."""
    p = Path(path)
    if p.is_dir():
        D = np.load(p / f"{array}.npy", mmap_mode="r"); t = np.load(p / "targets.npy").astype(str)
        g = np.load(p / "genes.npy").astype(str)
    else:
        z = np.load(p, allow_pickle=False); D = z["delta"]
        t = z["targets"].astype(str) if "targets" in z else np.array(targets)
        g = z["genes"].astype(str) if "genes" in z else np.array(genes)
    out = np.zeros((len(targets), len(genes)), np.float32)
    tp = {x: i for i, x in enumerate(t)}; gp = {x: i for i, x in enumerate(g)}
    gsrc = np.array([gp.get(x, -1) for x in genes]); ok = gsrc >= 0
    missing = 0
    for i, x in enumerate(targets):
        if x in tp:
            out[i, ok] = np.asarray(D[tp[x]])[gsrc[ok]]
        else:
            missing += 1
    if missing:
        print(f"  WARNING {path}: {missing} targets missing -> no change predicted for them")
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def kd_from_sources(targets, genes, orion_paths=(), tcell_path=None, tcell_condition="Stim48hr", pseudo=0.5):
    """Knocked-down gene's own log2 drop per target, averaged across sources (M1 rule). NaN if no source has it."""
    gp = {g: i for i, g in enumerate(genes)}; vals = [[] for _ in targets]
    for path in orion_paths:
        with h5py.File(path, "r") as f:
            obs = read_elem(f["obs"]); names = list(obs.index.astype(str)); n = obs["n_cells"].values.astype(float)
            var = read_elem(f["var"]); vg = list(var.index.astype(str))
            vpos = {g: i for i, g in enumerate(vg)}
            c = names.index(CONTROL)
            want = [(i, t) for i, t in enumerate(targets) if t in names and t in vpos]
            if not want:
                continue
            rows = sorted({c} | {names.index(t) for _, t in want})
            S = f["layers/sum_cpm"][rows, :]; rpos = {r: k for k, r in enumerate(rows)}
            ctrl = S[rpos[c]] / n[c]
            for i, t in want:
                r = names.index(t); col = vpos[t]
                vals[i].append(np.log2((S[rpos[r], col] / n[r] + pseudo) / (ctrl[col] + pseudo)))
    if tcell_path:
        t = ad.read_h5ad(tcell_path)
        if "condition" in t.obs:
            t = t[t.obs["condition"].astype(str) == tcell_condition]
        tg = t.obs["target_gene"].astype(str).values; tv = {g: i for i, g in enumerate(t.var_names.astype(str))}
        X = np.asarray(t.X)
        for i, x in enumerate(targets):
            r = np.flatnonzero(tg == x)
            if len(r) and x in tv and np.isfinite(X[r[0], tv[x]]):
                vals[i].append(float(X[r[0], tv[x]]))
    return np.array([np.mean(v) if v else np.nan for v in vals])


def kd_residual(log2_drop, cfg):
    r = np.where(np.isfinite(log2_drop), 2.0 ** np.nan_to_num(log2_drop), cfg.default_kd)
    return np.clip(r, *cfg.kd_clip)


# ---------------------------------------------------------------- core generator
def calibrate(D, line, targets, cfg):
    """Apply the single global alpha and a binary responder-gene expression gate.

    No further significance filtering, source scaling, or smooth knockdown gates.
    The +/-8 generator limit is a numerical safety cap, not an estimated prior.
    On-target suppression is handled separately by measured guide efficiency.
    """
    D = np.clip(cfg.alpha * D, -cfg.max_lfc, cfg.max_lfc).astype(np.float32)
    D[:, ~line.gate] = 0.0
    gp = {g: i for i, g in enumerate(line.genes)}
    for i, target in enumerate(targets):
        if target in gp: D[i, gp[target]] = 0.0
    return D


def pick_cells(line, d, cfg, rng):
    """Step 3 (+ optional tilt): which control cells become this knockdown's 400 cells."""
    n = cfg.n_cells
    if cfg.tilt and line.cyc is not None:
        dcc = float(np.mean(d[line.cc_idx]))                       # predicted cycling shift (log2)
        v = line.cyc - line.cyc.mean(); s = max(v.std(), 1e-6)
        if abs(dcc) > 0.02:
            lo, hi = -30.0, 30.0
            for _ in range(50):                                     # bisection: weighted mean shift == dcc
                b = (lo + hi) / 2; w = np.exp(np.clip(b * v / s, -50, 50)); m = (w * v).sum() / w.sum()
                lo, hi = (b, hi) if m < dcc else (lo, b)
            w = np.exp(np.clip(b * v / s, -50, 50)); w /= w.sum()
            k = min(n, int((w > 0).sum()))
            return rng.choice(line.n, k, replace=False, p=w) if k == n else rng.choice(line.n, n, replace=True, p=w)
    if n > line.n:
        order = np.concatenate([rng.permutation(g) for g in line.groups])
        return np.concatenate([order, rng.choice(line.n, n-line.n, replace=True)])
    per = int(np.ceil(n / len(line.groups)))
    pick = np.concatenate([rng.choice(v, size=min(per, len(v)), replace=False) for v in line.groups])
    if len(pick) >= n:
        return rng.choice(pick, n, replace=False)
    rest = np.setdiff1d(np.arange(line.n), pick)
    return np.concatenate([pick, rng.choice(rest, n - len(pick), replace=len(rest) < n - len(pick))])


def project_factors(share, delta, j=None, kd=None):
    """Project all-gene factors onto the composition simplex without an LFC floor.

    The realized expectation is log2(projected_factor), not the unconstrained
    delta. Zero-baseline induction uses the same 0.5-CPM pseudocount as labels.
    """
    fac = np.exp2(np.asarray(delta, float))
    if j is not None and kd is not None: fac[j] = kd
    proposed = np.asarray(share, float)*fac
    zero_induced = (share == 0) & (fac > 1)
    proposed[zero_induced] = 0.5e-6*(fac[zero_induced]-1)
    mass = float(proposed.sum())
    if not np.isfinite(mass) or mass <= 0: raise ValueError("invalid projected composition")
    projected = proposed/mass
    factors = np.divide(projected, share, out=np.ones_like(projected), where=share > 0)
    return factors, projected, mass


def generate_one(line, d, j, kd, cfg, rng=None, pick=None, line_key="line", target_key="target"):
    """Integer counts, with stable per-gene streams shared between perturbations."""
    if rng is None: rng = np.random.default_rng(seed_for(cfg.seed, line_key, target_key))
    if pick is None:
        # Same base pool per line; target order does not advance a line-level stream.
        pick = pick_cells(line, d, cfg, np.random.default_rng(seed_for(cfg.seed, line_key, "base_controls")))
    B = line.X[pick].toarray().astype(np.int64); d = np.asarray(d, float).copy()
    if cfg.match_pool:
        sampled = np.asarray(line.cpm_cells[pick].mean(0)).ravel()
        deviation = np.log2((sampled+0.5)/(line.cpm+0.5))
        mask = line.gate.copy()
        if j is not None: mask[j] = False
        d[mask] = np.clip(d[mask]-deviation[mask], -cfg.max_lfc, cfg.max_lfc)
    factors, projected, mass = project_factors(line.share, d, j, kd)
    # Library size is an independent whole-cell scale after composition projection.
    lib_scale = 1.0
    if cfg.library_size is not None:
        lib_scale = float(cfg.library_size)/max(float(np.median(line.lib[pick])), 1)
    positive = line.share > 0
    effective_pi = min(cfg.pi, float(factors[positive].min())) if positive.any() else 0.0
    esc_rng = np.random.default_rng(seed_for(cfg.seed, line_key, "escapers"))
    escapers = esc_rng.random(len(pick)) < effective_pi
    cols = np.flatnonzero((np.abs(factors*lib_scale-1) > 1e-12) | ((line.share == 0) & (projected > 0)))
    for g in cols:
        # Fresh per-gene seed gives common random draws without target-order dependence.
        grng = np.random.default_rng(seed_for(cfg.seed, line_key, "gene", line.genes[g]))
        factor = factors[g]
        inside = max((factor-effective_pi)/(1-effective_pi), 0)
        F = np.where(escapers, 1.0, inside)*lib_scale
        original = B[:, g].copy()
        values = grng.binomial(original, np.minimum(F, 1.0))
        up = np.maximum(F-1, 0)
        base = line.lib[pick]*line.share[g]
        if line.share[g] == 0 and projected[g] > 0:
            # Induced genes absent from controls must be able to gain counts.
            intensity = line.lib[pick]*projected[g]*lib_scale
        else:
            expected = cfg.up_w*original+(1-cfg.up_w)*base
            intensity = up*expected
        if np.any(intensity > 0):
            dispersion = float(line.overdispersion[g])
            if dispersion > 1e-8:
                intensity = grng.gamma(1/dispersion, intensity*dispersion)
            values += grng.poisson(intensity)
        B[:, g] = values
    if B.max(initial=0) > np.iinfo(np.int32).max: raise ValueError("count exceeds int32 range")
    out = sp.csr_matrix(B.astype(np.int32)); out.eliminate_zeros(); validate_counts(out)
    planned = np.log2((projected*1e6+0.5)/(line.share*1e6+0.5))
    return out, dict(net_share_change=mass-1, projection_normalizer=mass, n_changed=int(len(cols)),
                     effective_pi=effective_pi, library_scale=lib_scale), planned


def target_metadata(path, line, fallback_n, enabled=True):
    """Optional CSV metadata; supports shared or per-context rows/column suffixes."""
    df = pd.read_csv(path)
    if 'target_gene' not in df: raise ValueError("pert_counts.csv needs target_gene")
    if 'context' in df: df = df[df.context.astype(str) == str(line)]
    if df.target_gene.duplicated().any(): raise ValueError("duplicate target rows; supply context labels")
    result = {}
    for row in df.to_dict('records'):
        n, umi = fallback_n, None
        if enabled:
            for field, default in [('n_cells', fallback_n), ('median_umi_per_cell', None)]:
                value = row.get(field+'_'+str(line), row.get(field, default))
                if value is None or pd.isna(value): value = default
                if value is not None and (not np.isfinite(value) or value <= 0): raise ValueError(f"invalid {field} for {row['target_gene']}")
                if field == 'n_cells':
                    if float(value) != int(value): raise ValueError("n_cells must be integer")
                    n = int(value)
                else: umi = None if value is None else float(value)
        result[str(row['target_gene'])] = dict(n_cells=n, median_umi_per_cell=umi)
    if not result: raise ValueError(f"no requested targets for context {line}")
    return result


def check_output(path, expected_shape, block=512):
    a = ad.read_h5ad(path, backed='r')
    try:
        if a.shape != expected_shape: raise ValueError(f"output shape {a.shape} != {expected_shape}")
        stored = 0
        for start in range(0, a.n_obs, block):
            matrix = a.X[start:start+block]; validate_counts(matrix, f"output rows {start}:{start+block}")
            if np.any(np.asarray(matrix.sum(1)).ravel() > 1e6): raise ValueError('cell exceeds 1,000,000 count cap')
            stored += matrix.nnz if sp.issparse(matrix) else matrix.size
        if a.n_obs > 400000 or stored > 4750000000: raise ValueError('submission exceeds cell or stored-entry cap')
    finally: a.file.close()
    return True


def answer_free_check(cells, line, planned, j):
    pooled = np.asarray(cells.sum(0)).ravel(); pooled = pooled/max(pooled.sum(), 1)*1e6
    realized = np.log2((pooled+0.5)/(line.share*1e6+0.5))
    mask = np.isfinite(realized) & (line.share > 0)
    if j is not None: mask[j] = False
    return dict(projected_effect_mae=float(np.abs(realized[mask]-planned[mask]).mean()),
                n_cells=cells.shape[0], median_umi_per_cell=float(np.median(np.asarray(cells.sum(1)).ravel())))


def prediction_ontarget(path, targets):
    p = Path(path)
    if not p.is_dir() or not (p/'ontarget_lfc.npy').exists(): return np.full(len(targets), np.nan)
    names = np.load(p/'targets.npy').astype(str); values = np.load(p/'ontarget_lfc.npy'); pos = dict(zip(names, values))
    return np.asarray([pos.get(t, np.nan) for t in targets])


def cmd_submit(a):
    cfg = Cfg(a)
    if a.settings:
        settings = json.loads(Path(a.settings).read_text())
        if settings.get('schema') != 'vcc_2026_2_average_v1': raise ValueError('use generator settings from this notebook')
        chosen = settings['settings']
        for key in ['alpha', 'pi', 'up_w', 'n_cells', 'metadata', 'match_pool', 'tilt']:
            if key in chosen: setattr(cfg, key, chosen[key])
    data = Path(a.data); genes = read_list(data/'gene_names.csv', 'gene_name')
    gp = {g: i for i, g in enumerate(genes)}; deltas = dict(x.split('=', 1) for x in a.delta)
    specifications = {line: target_metadata(data/'pert_counts.csv', line, cfg.n_cells, cfg.metadata) for line in a.lines}
    sets = [set(s) for s in specifications.values()]
    if any(s != sets[0] for s in sets): raise ValueError('all submitted contexts must share the exact perturbation panel')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True); path = out/'prediction.h5ad'
    if path.exists(): raise FileExistsError('choose a new output directory to preserve the previous submission')
    for line_specs in specifications.values():
        if any(s['n_cells'] != a.required_n_cells for s in line_specs.values()):
            raise ValueError(f'2026 submissions require exactly {a.required_n_cells} cells per target; optional CSV metadata must respect that contract')
    frames = [pd.DataFrame(dict(target_gene=t, context=line), index=[f'{line}_{t}_{i:05d}' for i in range(spec['n_cells'])])
              for line in a.lines for t, spec in specifications[line].items()]
    obs = pd.concat(frames); obs['target_gene'] = obs.target_gene.astype('category'); obs['context'] = obs.context.astype('category')
    report = dict(settings=cfg.as_dict(), delta_array=a.delta_array, lines={})
    with h5py.File(path, 'w') as f:
        write_elem(f, 'obs', obs); write_elem(f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))
        f.attrs['encoding-type'] = 'anndata'; f.attrs['encoding-version'] = '0.1.0'; ds = None
        for name in a.lines:
            if name not in deltas: raise ValueError(f'no predictions for {name}')
            line = Line.from_h5ad(data/f'context_{name}.h5ad', genes)
            targets = list(specifications[name]); D = calibrate(load_deltas(deltas[name], targets, genes, a.delta_array), line, targets, cfg)
            own = prediction_ontarget(deltas[name], targets)
            if a.kd_orion or a.kd_tcell:
                legacy_own = kd_from_sources(targets, genes, a.kd_orion or (), a.kd_tcell, a.kd_tcell_condition)
                own = np.where(np.isfinite(own), own, legacy_own)
            kd = kd_residual(own, cfg); checks = []; diags = []
            # Pick max count once; different n_cells use prefixes of this same pool.
            pool_cfg = copy.copy(cfg); pool_cfg.n_cells = max(s['n_cells'] for s in specifications[name].values())
            common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(cfg.seed, name, 'base_controls')))
            for i, t in enumerate(targets):
                local = copy.copy(cfg); local.n_cells = specifications[name][t]['n_cells']; local.library_size = specifications[name][t]['median_umi_per_cell']
                pick = None if local.tilt else common[:local.n_cells]
                cells, diag, planned = generate_one(line, D[i], gp.get(t), kd[i], local, pick=pick, line_key=name, target_key=t)
                diags.append(dict(target=t, **diag))
                if i < a.checks: checks.append(dict(target=t, **answer_free_check(cells, line, planned, gp.get(t))))
                # float32 holds these integer counts exactly and matches vcc prep's output encoding,
                # so prep packages without a second full-matrix copy (int32 needs ~37 GB at 360k cells).
                if cells.data.max(initial=0) >= 2**24: raise ValueError('count too large for exact float32')
                cells = cells.astype(np.float32)
                if ds is None: write_elem(f, 'X', int64_indptr(cells)); ds = sparse_dataset(f['X'])
                else: ds.append(cells)
            pd.DataFrame(checks).to_csv(out/f'checks_{name}.csv', index=False)
            pd.DataFrame(diags).to_csv(out/f'projection_{name}.csv', index=False)
            report['lines'][name] = dict(n_targets=len(targets), n_cells=sum(s['n_cells'] for s in specifications[name].values()),
                metadata=specifications[name], measured_ontarget=int(np.isfinite(own).sum()))
            print(f"generated context {name}", flush=True)
    report['integer_counts'] = check_output(path, (len(obs), len(genes)))
    save_json(out/'generator_report.json', report); print(f'WROTE {path}; checked every output row')


class StreamingH5AD:
    """An .h5ad whose obs is known up front and whose X arrives one block at a time.

    Nothing but the current block is held in memory. The row pointer is int64 from the first
    block, so the finished file may exceed 2**31 stored values. Every block is cast to the first
    block's dtype, and the row total is checked against obs on close.
    """
    def __init__(self, path, obs, genes):
        self.path = Path(path); self.path.parent.mkdir(parents=True, exist_ok=True)
        self.n = len(obs); self.rows = 0; self.ds = None; self.dtype = None
        self.f = h5py.File(self.path, 'w')
        self.f.attrs['encoding-type'] = 'anndata'; self.f.attrs['encoding-version'] = '0.1.0'
        write_elem(self.f, 'obs', obs); write_elem(self.f, 'var', pd.DataFrame(index=pd.Index(genes, name='gene_name')))

    def append(self, block):
        block = block.tocsr() if sp.issparse(block) else sp.csr_matrix(np.asarray(block))
        if self.ds is None:
            self.dtype = block.dtype
            write_elem(self.f, 'X', int64_indptr(block)); self.ds = sparse_dataset(self.f['X'])
        else:
            self.ds.append(block.astype(self.dtype, copy=False))
        self.rows += block.shape[0]

    def close(self):
        self.f.close()
        if self.rows != self.n: raise ValueError(f'{self.path}: wrote {self.rows} rows for {self.n} obs entries')


def labelled_obs(labels, prefix):
    labels = np.asarray(labels, dtype=str)
    return pd.DataFrame({'target_gene': pd.Categorical(labels)}, index=[f'{prefix}{i:07d}' for i in range(len(labels))])


def stream_h1(a, genes, real_path, line_pseudo=0.5, block=4096):
    """Write the scoring reference straight from the backed H1 file, one target at a time.

    Controls are split into two disjoint, seeded halves: the first half becomes the generator's
    input (held in memory, it is sampled for every target); the second half is appended to the
    reference after the perturbed cells. In oracle mode each target's cells are split the same
    way as before: the estimation half only produces that target's effect vector, the other half
    is appended to the reference. Peak memory is the control half plus one target.

    Returns (Line, targets, {target: reference cell count}, oracle effects or None, oracle own LFCs).
    """
    h = ad.read_h5ad(a.cells, backed='r')
    try:
        labels = h.obs[a.obs_col].astype(str).values
        requested = sorted(set(read_list(a.targets, 'target_gene'))) if a.targets else sorted(set(labels) - {CONTROL})
        counts = pd.Series(labels).value_counts()
        targets = [t for t in requested if counts.get(t, 0) >= a.min_cells and t != CONTROL]
        if a.n_targets and len(targets) > a.n_targets:
            targets = sorted(np.random.default_rng(seed_for(a.seed, 'H1', 'panel')).choice(targets, a.n_targets, replace=False).tolist())
        if not targets: raise ValueError('no eligible H1 targets')

        def subset(rows):
            rows = np.sort(np.asarray(rows))
            small = h[rows].to_memory(); validate_counts(small.X, 'H1 raw counts')
            return aligned_X(small, genes)

        ci = np.flatnonzero(labels == CONTROL)
        if len(ci) < 2: raise ValueError('H1 needs controls')
        ci = np.random.default_rng(seed_for(a.seed, 'H1', 'controls')).permutation(ci)
        train_rows, score_rows = ci[:len(ci) // 2], ci[len(ci) // 2:]
        line = Line(subset(train_rows), None, genes)
        # The scorer uses its REAL control half. Match that exact gate, even when
        # the generator half happens to put a near-threshold gene on the other side.
        score_stats = moments(h, np.sort(score_rows), alignment(h.var_names.astype(str), genes))
        line.gate = expression_gate(score_stats['mean_cpm'])
        save_json(Path(real_path).parent / 'control_split.json', dict(seed=a.seed,
                  generator_controls=len(train_rows), scoring_controls=len(score_rows),
                  gate_source='scoring real controls', genes_above_5_cpm=int(line.gate.sum())))
        plan = {}
        for t in targets:
            rows = np.flatnonzero(labels == t)
            if a.mode == 'oracle':
                rows = np.random.default_rng(seed_for(a.seed, 'H1', t, 'split')).permutation(rows)
                plan[t] = (rows[:len(rows) // 2], rows[len(rows) // 2:])
            else:
                plan[t] = (None, rows)
        real_counts = {t: len(ref) for t, (_, ref) in plan.items()}
        obs_labels = [t for t in targets for _ in range(real_counts[t])] + [CONTROL] * len(score_rows)
        writer = StreamingH5AD(real_path, labelled_obs(obs_labels, 'real_'), genes)
        oracle, own = ([], []) if a.mode == 'oracle' else (None, None)
        gp = {g: i for i, g in enumerate(genes)}
        try:
            for i, t in enumerate(targets):
                estimation, reference = plan[t]
                if estimation is not None:
                    x = subset(estimation); lib = np.asarray(x.sum(1)).ravel()
                    mean = np.asarray((sp.diags(1e6 / np.maximum(lib, 1)) @ x).mean(0)).ravel()
                    d = np.log2((mean + line_pseudo) / (line.cpm + line_pseudo)); oracle.append(d)
                    own.append(d[gp[t]] if t in gp else np.nan); del x
                writer.append(subset(reference))
                print(f'  reference: {i + 1}/{len(targets)} targets', end='\r', flush=True)
            sorted_score = np.sort(score_rows)
            for start in range(0, len(sorted_score), block):
                writer.append(subset(sorted_score[start:start + block]))
        finally:
            writer.close()
        print(f'\n  wrote {real_path}: {writer.rows:,} cells ({len(score_rows):,} scoring controls)', flush=True)
        return (line, targets, real_counts, None if oracle is None else np.asarray(oracle),
                None if own is None else np.asarray(own))
    finally:
        h.file.close()


def write_arm(path, line, effects, kd, targets, n_cells, common, cfg, line_key='H1', block=4096):
    """Generate each target's cells and append them immediately, then the generator's control cells.

    cell-eval2 requires the predicted file to carry the control label even though vcc2026 scores
    every perturbation against the REAL controls (`control_source: real`), so the generator's own
    input controls are appended as before; they are not used as the scoring reference.
    """
    gp = {g: i for i, g in enumerate(line.genes)}
    labels = [t for t in targets for _ in range(n_cells[t])] + [CONTROL] * line.n
    writer = StreamingH5AD(path, labelled_obs(labels, 'pred_'), line.genes)
    try:
        for i, t in enumerate(targets):
            local = copy.copy(cfg); local.n_cells = n_cells[t]
            cells = generate_one(line, effects[i], gp.get(t), kd[i], local, pick=common[:local.n_cells],
                                 line_key=line_key, target_key=t)[0]
            writer.append(cells)
        for start in range(0, line.n, block):
            writer.append(line.X[start:start + block].astype(np.int32))
    finally:
        writer.close()


class OfficialEvaluator:
    """Runs official cell-eval2 against one reference file.

    The reference never changes between arms, so its expensive side (real DE tests,
    pseudobulks, noise corrections) is cached once with `prep-cache --side real` and reused by
    every arm through `--cache-real`. The ceiling depends on the real data only, so it is computed
    once in real-only mode instead of inside every arm.
    """
    def __init__(self, a, reference, out):
        self.binary = shutil.which(a.cell_eval)
        if self.binary is None: raise RuntimeError('install official cell-eval2 (pip install -r requirements-eval.txt) before running the H1 benchmark')
        help_text = subprocess.run([self.binary, 'run', '--help'], check=True, text=True, capture_output=True).stdout
        if '--ceiling' not in help_text or '--cache-real' not in help_text:
            raise RuntimeError('installed cell-eval2 lacks --ceiling/--cache-real; install requirements-eval.txt')
        self.a, self.reference, self.out = a, Path(reference), Path(out)
        self.cache = self.out / 'real_cache'
        self.common = ['--pert-col', 'target_gene', '--control', CONTROL, '--profile', a.eval_profile,
                       '--input-type', 'counts', '--preset', 'vcc2026', '--set', 'de.backend=pdex',
                       '--set', f'num_threads={a.num_threads}']
        self.invocations = []

    def _run(self, command):
        print('  $ ' + ' '.join(shlex.quote(str(x)) for x in command), flush=True)
        subprocess.run([str(x) for x in command], check=True); self.invocations.append([str(x) for x in command])

    def prepare(self):
        # One-sided cache warm: the expression comparator must be named. vcc2026 on counts resolves
        # to bulk_lognorm (cell_eval2.competition.competition_comparator()).
        self._run([self.binary, 'prep-cache', '--side', 'real', '--adata', self.reference,
                   '--cache-real', self.cache, '--comparator', 'bulk_lognorm', *self.common])
        self._run([self.binary, 'run', '-ar', self.reference, '-o', self.out / 'ceiling', '--ceiling',
                   '--cache-real', self.cache, *self.common])

    def score(self, prediction, output):
        self._run([self.binary, 'run', '-ap', prediction, '-ar', self.reference, '-o', output,
                   '--cache-real', self.cache, *self.common])
        self.record(Path(output) / 'invocation.json')

    def record(self, path):
        import importlib.metadata
        versions = {}
        for package in ['cell-eval2', 'pdex', 'numpy', 'scipy', 'anndata']:
            try: versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: versions[package] = 'not installed in generator environment'
        save_json(path, dict(commands=self.invocations, versions=versions))


def metric_rank_table(outputs, directions_path):
    """Rank only explicitly requested metric families, with explicit directions.

    The evaluator emits diagnostics as well as scoring metrics. Never average
    their ranks blindly. Supply metric,direction ('min'/'max') from current rules.
    """
    directions = pd.read_csv(directions_path)
    if not {'metric', 'direction'} <= set(directions) or directions.metric.duplicated().any():
        raise ValueError('directions CSV needs unique metric,direction columns')
    values = []
    for label, path in outputs.items():
        df = pd.read_csv(Path(path)/'agg_results.csv')
        if not {'metric', 'value'} <= set(df):
            if {'metric', 'score'} <= set(df): df = df.rename(columns={'score': 'value'})
            elif set(directions.metric) <= set(df.columns):
                # cell-eval2 writes a wide describe-style frame; use only its mean row.
                mean_row = np.zeros(len(df), bool)
                for col in df.columns:
                    if col not in set(directions.metric): mean_row |= df[col].astype(str).eq('mean').to_numpy()
                if not mean_row.any() and len(df) == 1: mean_row[:] = True
                if mean_row.sum() != 1: raise ValueError('official aggregate CSV needs exactly one mean row')
                df = pd.DataFrame({'metric': directions.metric, 'value': [df.loc[mean_row, m].iloc[0] for m in directions.metric]})
            else: raise ValueError(f'unrecognized official aggregate schema: {df.columns.tolist()}')
        for metric, direction in directions[['metric', 'direction']].itertuples(index=False, name=None):
            v = df.loc[df.metric == metric, 'value']
            if len(v) != 1 or not np.isfinite(v.iloc[0]): raise ValueError(f'{label}: metric {metric} missing/nonfinite')
            if direction not in ['min', 'max']: raise ValueError('metric direction must be min/max')
            values.append(dict(variant=label, metric=metric, value=float(v.iloc[0]), direction=direction))
    frame = pd.DataFrame(values); frame['rank'] = np.nan
    for metric in directions.metric:
        use = frame.metric == metric
        frame.loc[use, 'rank'] = frame.loc[use, 'value'].rank(ascending=frame.loc[use, 'direction'].iloc[0] == 'min', method='average')
    return frame, frame.groupby('variant')['rank'].mean().sort_values(kind='stable')


def cmd_benchmark(a):
    """Official H1 benchmark, streamed: reference and each arm go to disk one target at a time.

    Oracle mode never writes selected generator settings.
    """
    genes = read_list(a.genes, 'gene_name')
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    if a.mode == 'transfer':
        provenance = json.loads((Path(a.predictions)/'prediction_report.json').read_text())
        if provenance.get('heldout_background') != 'H1': raise ValueError('generator tuning requires LOBO-H1 predictions (H1 excluded)')
        if provenance.get('schema') != 'vcc_2026_2_average_v1':
            raise ValueError('rebuild H1 predictions with vcc_2026_2; old shrunken predictions are incompatible')
        if 'H1' in provenance.get('source_backgrounds', []): raise ValueError('H1 leaked into transfer sources')
        if a.delta_array == 'selected': a.delta_array = provenance['selected_method']
        if a.delta_array != 'source_mean': raise ValueError('this notebook benchmarks the fixed source average')
    real_path = out/'real.h5ad'
    line, targets, real_counts, oracle_effects, oracle_own = stream_h1(a, genes, real_path)
    evaluator = OfficialEvaluator(a, real_path, out); evaluator.prepare()
    if a.mode == 'transfer':
        methods = [a.delta_array]  # one fixed model; no duplicate ridge/baseline arms
        kd = kd_residual(prediction_ontarget(a.predictions, targets), Cfg(a))
        effects = {method: load_deltas(a.predictions, targets, genes, method) for method in methods}
        variants = [dict(alpha=alpha, pi=pi, up_w=w, cell_rule=rule)
                    for alpha in a.alpha_grid for pi in a.pi_grid for w in a.up_w_grid for rule in a.cell_rules]
    else:
        # Ceiling-only oracle inputs: per-cell arithmetic CPM from each target's estimation half.
        effects = {'oracle': oracle_effects}; kd = kd_residual(oracle_own, Cfg(a))
        variants = [dict(alpha=a.alpha, pi=a.pi, up_w=a.up_w, cell_rule='matched')]
    outputs, settings = {}, {}
    maximum = max([a.n_cells] + list(real_counts.values()))
    pool_cfg = Cfg(a); pool_cfg.n_cells = maximum
    common = pick_cells(line, np.zeros(len(genes)), pool_cfg, np.random.default_rng(seed_for(a.seed, 'H1', 'base_controls')))
    for method, effects_array in effects.items():
        for index, variant in enumerate(variants):
            label = f'{method}_{index:03d}'
            cfg = Cfg(argparse.Namespace(**dict(vars(a), **{k: v for k, v in variant.items() if k != 'cell_rule'})))
            calibrated = calibrate(effects_array, line, targets, cfg)
            n_cells = {t: (real_counts[t] if variant['cell_rule'] == 'matched' else a.n_cells) for t in targets}
            pred_path = out/f'{label}.h5ad'
            write_arm(pred_path, line, calibrated, kd, targets, n_cells, common, cfg)
            del calibrated
            evaluator.score(pred_path, out/label)
            outputs[label] = str(out/label); settings[label] = dict(method=method, **variant)
            save_json(out/f'{label}.json', settings[label])
            print(f'official evaluation complete: {label}', flush=True)
    save_json(out/'benchmark.json', dict(mode=a.mode, outputs=outputs, settings=settings, n_targets=len(targets),
              ceiling=str(out/'ceiling'), real_cache=str(evaluator.cache),
              note='H1 is generator calibration; final evidence requires unseen lines. Oracle is a generator-only ceiling check.'))
    if a.metric_directions or a.eval_profile == 'vcc2026':
        directions = a.metric_directions or str(out/'metric_directions.csv')
        if a.metric_directions is None:
            pd.DataFrame({'metric': list(VCC2026_METRICS), 'direction': list(VCC2026_METRICS.values())}).to_csv(directions, index=False)
        frame, ranks = metric_rank_table(outputs, directions)
        frame.to_csv(out/'metric_ranks.csv', index=False); ranks.to_csv(out/'aggregate_rank.csv')
        if a.mode == 'transfer':
            # Only global generator alpha is varied by the notebook; the source average is fixed.
            eligible = [label for label in ranks.index if settings[str(label)]['method'] == a.delta_array and settings[str(label)]['cell_rule'] == 'fixed']
            if not eligible: raise ValueError('include fixed cell rule to select submission-compatible generator settings')
            best = settings[str(eligible[0])]
            save_json(out/'selected_generator.json', dict(settings=dict({k:v for k,v in best.items() if k not in ['method', 'cell_rule']}, n_cells=a.n_cells),
                       cell_rule='fixed', n_cells=a.n_cells, diagnostic_best_method=best['method'],
                       delta_array='selected', selection='official metric average rank on LOBO-H1 transfer predictions',
                       schema='vcc_2026_2_average_v1', note='fixed equal-background average; global alpha only; H1 is calibration, not an untouched test'))


def add_switches(p):
    p.add_argument('--alpha', type=float, default=1.0); p.add_argument('--n-cells', type=int, default=400)
    p.add_argument('--pi', type=float, default=0); p.add_argument('--up-w', type=float, default=0.75)
    p.add_argument('--metadata', action=argparse.BooleanOptionalAction, default=True)
    p.add_argument('--match-pool', action='store_true'); p.add_argument('--tilt', action='store_true')
    p.add_argument('--seed', type=int, default=2026)


if __name__ == '__main__':
    p = argparse.ArgumentParser(description=__doc__); sub = p.add_subparsers(dest='cmd', required=True)
    s = sub.add_parser('submit'); add_switches(s)
    for key in ['data', 'out']: s.add_argument('--'+key, required=True)
    s.add_argument('--lines', nargs='+', default=['A', 'B', 'C']); s.add_argument('--delta', action='append', required=True)
    s.add_argument('--required-n-cells', type=int, default=400)
    s.add_argument('--delta-array', default='selected'); s.add_argument('--settings')
    s.add_argument('--kd-orion', nargs='*'); s.add_argument('--kd-tcell'); s.add_argument('--kd-tcell-condition', default='Stim48hr')
    s.add_argument('--checks', type=int, default=20)
    for command, mode in [('benchmark', 'transfer'), ('oracle', 'oracle')]:
        b = sub.add_parser(command); add_switches(b); b.set_defaults(mode=mode)
        for key in ['cells', 'genes', 'out']: b.add_argument('--'+key, required=True)
        b.add_argument('--predictions', required=mode == 'transfer'); b.add_argument('--targets')
        b.add_argument('--obs-col', default='target_gene'); b.add_argument('--delta-array', default='selected')
        b.add_argument('--min-cells', type=int, default=100); b.add_argument('--n-targets', type=int, default=0)
        b.add_argument('--alpha-grid', nargs='+', type=float, default=[0.75, 1.0, 1.25])
        b.add_argument('--pi-grid', nargs='+', type=float, default=[0.0])
        b.add_argument('--up-w-grid', nargs='+', type=float, default=[0.75])
        b.add_argument('--cell-rules', nargs='+', choices=['matched', 'fixed'], default=['fixed'])
        b.add_argument('--cell-eval', default='cell-eval2'); b.add_argument('--eval-profile', default='vcc2026')
        b.add_argument('--num-threads', type=int, default=4)
        b.add_argument('--metric-directions', help='CSV metric,direction for current official scoring metrics only')
    a = p.parse_args()
    if a.cmd == 'submit': cmd_submit(a)
    else: cmd_benchmark(a)


## 2. Public effects and coverage
Existing unfiltered effect tables can be reused without re-running large raw-cell jobs. The configuration is always rebuilt with the new fixed-average schema. Set `REUSE_EXISTING_EFFECTS = False` to build fresh tables. Background processes must finish successfully before running dependent cells.

Use the existing merged H1 file, `lib/h1/H1_full_2025.h5ad`, directly via `H1_CELLS`. No H1 merge step is needed.


In [ ]:
import json
import numpy as np
import pandas as pd
base_contexts = ['ORION_HCT116', 'ORION_HEK293T', 'TCELL_Stim48hr', 'TCELL_Rest', 'ARC_H1']
required = ['context.json', 'meta.json', 'lfc.npy', 'se.npy', 'targets.npy', 'genes.npy', 'control_profile.npz']
new_root = Path(ENV['WORK'])
existing_root = new_root if all((new_root / 'effects' / name / f).exists() for name in base_contexts for f in required) else EXISTING_EFFECTS
existing = [existing_root / 'effects' / name for name in base_contexts]
if REUSE_EXISTING_EFFECTS and all((d / f).exists() for d in existing for f in required):
    dirs = ' '.join(shlex.quote(str(d)) for d in existing)
    run(f'"$PY" vcc2_build_effects.py make-config --genes "$DATA/gene_names.csv" --dirs {dirs} --out "$WORK/contexts_initial.json"')
    run('"$PY" vcc2_transfer.py coverage --config "$WORK/contexts_initial.json" --targets "$DATA/pert_counts.csv" --out "$WORK/coverage_initial"')
    extra = sorted(d for d in (existing_root / 'replogle_effects').glob('*') if all((d / f).exists() for f in required))
    dirs = ' '.join(shlex.quote(str(d)) for d in existing + extra)
    neighbours = f'--neighbours {shlex.quote(str(NEIGHBOURS))}' if NEIGHBOURS.exists() else ''
    run(f'"$PY" vcc2_build_effects.py make-config --genes "$DATA/gene_names.csv" --dirs {dirs} {neighbours} --out "$WORK/contexts.json"')
    run('"$PY" vcc2_transfer.py coverage --config "$WORK/contexts.json" --targets "$DATA/pert_counts.csv" --out "$WORK/coverage_expanded"')
    print('Reused unfiltered effect tables and profiles:', existing + extra)
else:
    bg('effects', r"""
    mkdir -p "$WORK/effects"
    "$PY" vcc2_build_effects.py orion --pseudobulk "$LIB/orion/HCT116_pseudobulk.h5ad" --genes "$DATA/gene_names.csv" --name ORION_HCT116 --background HCT116 --out "$WORK/effects/ORION_HCT116"
    "$PY" vcc2_build_effects.py orion --pseudobulk "$LIB/orion/HEK293T_pseudobulk.h5ad" --genes "$DATA/gene_names.csv" --name ORION_HEK293T --background HEK293T --out "$WORK/effects/ORION_HEK293T"
    "$PY" vcc2_build_effects.py tcell --effects "$LIB/tcell/TCELL_Stim48hr_effects.h5ad" --genes "$DATA/gene_names.csv" --name TCELL_Stim48hr --priority 1 --out "$WORK/effects/TCELL_Stim48hr"
    "$PY" vcc2_build_effects.py tcell --effects "$LIB/tcell/TCELL_Rest_effects.h5ad" --genes "$DATA/gene_names.csv" --name TCELL_Rest --out "$WORK/effects/TCELL_Rest"
    "$PY" vcc2_build_effects.py cells --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --obs-col target_gene --control non-targeting --kind h1 --name ARC_H1 --background H1 --platform flex --out "$WORK/effects/ARC_H1"
    DIRS=("$WORK/effects/ORION_HCT116" "$WORK/effects/ORION_HEK293T" "$WORK/effects/TCELL_Stim48hr" "$WORK/effects/TCELL_Rest" "$WORK/effects/ARC_H1")
    "$PY" vcc2_build_effects.py make-config --genes "$DATA/gene_names.csv" --dirs "${DIRS[@]}" --out "$WORK/contexts_initial.json"
    "$PY" vcc2_transfer.py coverage --config "$WORK/contexts_initial.json" --targets "$DATA/pert_counts.csv" --out "$WORK/coverage_initial"
    if [ -f "$REPLOGLE_MANIFEST" ]; then
      "$PY" vcc2_build_effects.py manifest --manifest "$REPLOGLE_MANIFEST" --genes "$DATA/gene_names.csv" --out "$WORK/replogle_effects"
      for D in "$WORK"/replogle_effects/*; do [ -f "$D/context.json" ] && DIRS+=("$D"); done
    else
      echo "Replogle manifest missing: supply lib/replogle/manifest.csv to expand coverage."
    fi
    EXTRA=()
    if [ -f "$NEIGHBOURS" ]; then EXTRA+=(--neighbours "$NEIGHBOURS"); fi
    "$PY" vcc2_build_effects.py make-config --genes "$DATA/gene_names.csv" --dirs "${DIRS[@]}" "${EXTRA[@]}" --out "$WORK/contexts.json"
    "$PY" vcc2_transfer.py coverage --config "$WORK/contexts.json" --targets "$DATA/pert_counts.csv" --out "$WORK/coverage_expanded"
    """)


In [ ]:
status('effects')  # No background log is expected when existing tables were reused.


## 3. Current context profiles
Build A/B/C profiles from their real controls with the existing pooled-CPM representation. The coverage report records profile correlations for inspection. These diagnostics do not set weights in the current equal-background average.


In [ ]:
for line in LINES:
    run(f'"$PY" vcc2_context_profile.py --counts "$DATA/context_{line}.h5ad" --all-controls --genes "$DATA/gene_names.csv" --out "$WORK/profiles/{line}.npz"')
profile_args = ' '.join(f'--profile {shlex.quote(line + "=" + str(Path(ENV["WORK"]) / "profiles" / (line + ".npz")))}' for line in LINES)
run(f'"$PY" vcc2_transfer.py coverage --config "$WORK/contexts.json" --targets "$DATA/pert_counts.csv" --out "$WORK/coverage_lines" {profile_args}')


### Deferred: context-aware precision weighting and power-based DE
The next cell is intentionally blank. Future work: improve context profiling; consider `w = 1 / (SE² + τ²)` with lineage-specific `τ²`, and power-based DE calls from each line's own control counts. Neither is implemented in this baseline. A possible later 2–5-feature gene-level correction (expression, dispersion, distance from 5 CPM) must be assessed with `cell-eval2` on H1, not fold-change MAE.


## 4. Prepare the fixed average and H1 holdout
There is no transfer fitting or hyperparameter search. Prepare H1-excluded sources and the all-source final model, then make fresh H1 predictions. The held-out H1 effect table is used only to identify an evaluation panel; H1 perturbation outcomes never enter its source average. Its untreated profile is permitted context information.

Panels retain knockdowns measured in another background and are selected reproducibly. Other biological holdouts can be inspected with the same `prepare --holdout` command, but do not select transfer settings. Old checkpoints and prediction reports are incompatible with the new schema.


In [ ]:
config = json.loads((Path(ENV['WORK']) / 'contexts.json').read_text())
h1_spec = next(c for c in config['contexts'] if c['name'] == 'ARC_H1')
ENV['H1_PROFILE'] = h1_spec['profile']
os.environ['H1_PROFILE'] = ENV['H1_PROFILE']
run('"$PY" vcc2_transfer.py panels --config "$WORK/contexts.json" --out "$WORK/panels" --max-targets 1000 --seed 2026')
bg('average', r"""
"$PY" vcc2_transfer.py prepare --config "$WORK/contexts.json" --holdout H1 --out "$WORK/models/holdout_H1"
"$PY" vcc2_transfer.py predict --model "$WORK/models/holdout_H1" --profile "$H1_PROFILE" --targets "$WORK/panels/ARC_H1.csv" --out "$WORK/eval/H1/pred"
"$PY" vcc2_transfer.py prepare --config "$WORK/contexts.json" --out "$WORK/models/final"
""")


In [ ]:
status('average')


In [ ]:
report = json.loads((Path(ENV['WORK']) / 'models/final/model_report.json').read_text())
print(json.dumps({key: report[key] for key in ['weighting', 'signal_filter', 'expression_gate', 'fitted_parameters', 'source_backgrounds']}, indent=2))


## 5. Predict the challenge lines
Write the once-filtered source averages and per-gene source counts. `selected.npy` is an explicit alias of `source_mean.npy`; neither is a ridge prediction. The count generator applies the binary expression gate later, so the same average can be gated against the precise H1 scoring controls. Predictions refuse nonempty output folders.

The existing notebook's explicit no-change fallback is retained for unresolved targets and recorded in the coverage report. Set `ALLOW_UNSUPPORTED = False` to stop instead. The fallback also reports targets whose sources have no usable gene-level uncertainty; it does not invent source effects.


In [ ]:
ALLOW_UNSUPPORTED = True  # preserve the existing documented zero fallback (e.g. C5orf22)
fallback = '--allow-unsupported' if ALLOW_UNSUPPORTED else ''
for line in LINES:
    run(f'"$PY" vcc2_transfer.py predict --model "$WORK/models/final" --profile "$WORK/profiles/{line}.npz" --targets "$DATA/pert_counts.csv" {fallback} --out "$WORK/pred/{line}"')


## 6. Official H1 calibration
Use fresh H1-excluded predictions. The benchmark splits H1 controls reproducibly: one half supplies generation cells, the other supplies the real scoring reference and the strict expression gate. Perturbed H1 cells are seen only by the evaluator. An optional oracle estimates effects from separate perturbation halves and never chooses generator settings.

The fixed source average is evaluated once per global α value (`0.75, 1.0, 1.25`). Keep `pi=0`, `up_w=0.75`, and 400 predicted cells fixed. Compare all six official metrics; their average rank is a local calibration heuristic, not the leaderboard's reference scaling. No transfer threshold, source weight, source scale, ridge penalty, or profile parameter is tuned here.

Start with 25 reproducibly sampled eligible targets because official evaluation can require substantial memory. Increase only with sufficient memory; inspect metric trade-offs before interpreting small rank differences. No biological score or improvement is assumed in advance.


In [ ]:
BENCHMARK_TARGETS = 25
bg('benchmark_h1', f"""
"$PY" vcc2_generate.py benchmark --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/panels/ARC_H1.csv" --predictions "$WORK/eval/H1/pred" --delta-array source_mean --eval-profile vcc2026 --cell-rules fixed --n-targets {BENCHMARK_TARGETS} --alpha-grid 0.75 1.0 1.25 --pi-grid 0 --up-w-grid 0.75 --out "$WORK/benchmark_h1"
""")


In [ ]:
status('benchmark_h1', n=30)
settings = Path(ENV['WORK']) / 'benchmark_h1/selected_generator.json'
if settings.exists():
    print(settings.read_text())
    display(pd.read_csv(settings.parent / 'metric_ranks.csv'))


In [ ]:
RUN_ORACLE = False
if RUN_ORACLE:
    bg('oracle_h1', f'"$PY" vcc2_generate.py oracle --cells "$H1_CELLS" --genes "$DATA/gene_names.csv" --targets "$WORK/panels/ARC_H1.csv" --eval-profile vcc2026 --n-targets {BENCHMARK_TARGETS} --out "$WORK/oracle_h1"')


## 7. Generate and validate counts
Use the fixed all-source average and H1-selected global α. The generator gates responder genes at mean per-cell CPM >5 in each line's real controls, then projects composition and generates raw counts with the existing sampling and dispersion machinery. Direct on-target knockdown is separate from downstream transfer. A retained gene receives no additional significance shrinkage.

Optional library-depth metadata remains independent of effect scaling. Every output row is checked for finite, nonnegative integral counts, dimensions, and the cell-total cap. Use a new `RUN_NAME` for another run; existing results are preserved.


In [ ]:
settings = Path(ENV['WORK']) / 'benchmark_h1/selected_generator.json'
if not settings.exists(): raise FileNotFoundError('Run the official H1 benchmark before generation.')
result = V / 'results' / RUN_NAME
if result.exists() and any(result.iterdir()): raise FileExistsError('Choose a new RUN_NAME to preserve existing results.')
line_args = ' '.join(shlex.quote(line) for line in LINES)
delta_args = ' '.join(f'--delta {shlex.quote(line + "=" + str(Path(ENV["WORK"]) / "pred" / line))}' for line in LINES)
bg('generate', f'"$PY" vcc2_generate.py submit --data "$DATA" --lines {line_args} --delta-array source_mean --settings {shlex.quote(str(settings))} {delta_args} --out {shlex.quote(str(result))}')


In [ ]:
status('generate')


## 8. Package locally; optional submission and backup
Packaging validates the generated file with the installed VCC CLI. Remote submission and bucket backup are disabled until their explicit switches are enabled. After a kernel restart, rerun setup and set `RUN_NAME` to the run being packaged.


In [ ]:
SUBMISSION = V / 'results' / RUN_NAME
run(f'vcc prep {shlex.quote(str(SUBMISSION / "prediction.h5ad"))} -g "$DATA/gene_names.csv" --perts "$DATA/pert_counts.csv" -o {shlex.quote(str(SUBMISSION / "prediction.vcc"))}')


In [ ]:
SUBMIT_TO_LEADERBOARD = False
if SUBMIT_TO_LEADERBOARD:
    message = f'{RUN_NAME}: equal-background averaging, one z filter, binary CPM gate'
    run(f'vcc submit {shlex.quote(str(SUBMISSION / "prediction.vcc"))} -m {shlex.quote(message)} --wait')


In [ ]:
BACK_UP_TO_BUCKET = False
if BACK_UP_TO_BUCKET:
    run(r"""
    gcloud storage rsync -r "$WORK" "$B/lib/vcc_2026_2" --exclude='.*\.h5ad$'
    gcloud storage rsync -r "$CODE" "$B/code/vcc_2026_2"
    gcloud storage cp ~/vcc/vcc_2026_2.ipynb "$B/code/vcc_2026_2.ipynb"
    """)
